# TTS_0.10: VITS-style full training with switchable components

TTS_0.10 is benlu's `final-vits` model and training recipe, integrated with the project's
training harness (evaluation, checkpoints with optimizer state, resume, time budget, run
records) and TTS_0.02 speaker-ID conditioning.

Every component that differs from TTS_0.02 is a **switch**. An ablation arm reverts
exactly one switch to its TTS_0.02 implementation (`E0` reverts all of them), so the arms
measure which changes matter. Select an arm with `STS_ARM` (default `R0`, the complete
TTS_0.10). The arms, owners and records are described in `docs/ABLATION_TTS_0.10.md`.

| Switch | TTS_0.10 (default) | TTS_0.02 |
|---|---|---|
| `text_encoder` | Transformer blocks with a convolutional feed-forward network, scaled embeddings | `nn.TransformerEncoder`, GELU feed-forward |
| `posterior_encoder` | 513→256→192 stem, 8 gated WaveNet residual blocks with skip sum, log-scale clamp | 4 dilated convolutions |
| `decoder` | multi-receptive-field stages (kernels 3 and 7) | one residual block per stage |
| `discriminator` | multi-period (2, 3, 5, 7, 11) + one scale discriminator | one 1-D convolution stack |
| `feature_matching` | on, weight 2 | off |
| `recon_loss` | Slaney log-mel L1 × 45 | mel 1 + multi-resolution STFT 0.5 + waveform L1 10 |
| `optim` | batch 16, learning rate 2e-4, fp16 autocast, no gradient clipping | batch 8, learning rate 1e-4, fp32, clipping at 1.0 |
| `max_text_chars` | 100 | no length filter |

Shared by every arm: flow, duration predictor, duration and KL losses, LSGAN adversarial
loss (weight 1), AdamW (betas 0.8/0.99, eps 1e-9, weight decay 0.01), 40,000 steps,
batch-vectorised MAS, `csv.QUOTE_NONE` metadata parsing, the VITS magnitude spectrogram
`sqrt(|X|² + 1e-6)`, and exclusion of the 100 held-out clips in
`sts_eval/splits/ljspeech_heldout_100.txt`.

Scope: the in-run probes are short fixed crops. The eight LJ001 probe clips are a mix of
training clips and clips removed by the length filter; the held-out probe uses held-out
clips. Neither probe measures text-to-speech quality. Full held-out evaluation runs
afterwards on the saved checkpoints (`tools/eval_tts010.py`).

Optional and off by default: chunk-aware training (section 4b) and starting a new run from a
finished checkpoint (section 15), used together to fine-tune for chunked synthesis.

Speaker-ID mode (TTS_0.02) is unchanged: set `STS_SPEAKER_MANIFEST` and `STS_DATA_DIR`.
See `docs/SPEAKER_CONDITIONING.md`.

In [ ]:
import csv
import functools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.nn.utils import spectral_norm, weight_norm
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

## 1. Configuration, ablation arm and hardware

`STS_ARM` selects one arm. Its switches, seed and optimisation preset are resolved here and
recorded in `run_config.json`; nothing else differs between arms. All arms train for the same
number of optimizer steps, so arms with batch 8 (`optim = tts002`) see half as many clips.

One spectrogram frame corresponds to `HOP_LENGTH = 256` waveform samples. A training segment
contains 32 latent frames and produces 8,192 audio samples.

In [ ]:
import os
import random

MODEL_VERSION = "TTS_0.10"

# TTS_0.10 values. Each switch can revert one component to its TTS_0.02 implementation.
TTS010_SWITCHES = {
    "text_encoder": "tts010",
    "posterior_encoder": "tts010",
    "decoder": "tts010",
    "discriminator": "tts010",
    "feature_matching": True,
    "recon_loss": "tts010",
    "optim": "tts010",
    "max_text_chars": 100,
}
TTS002_SWITCHES = {
    "text_encoder": "tts002",
    "posterior_encoder": "tts002",
    "decoder": "tts002",
    "discriminator": "tts002",
    "feature_matching": False,
    "recon_loss": "tts002",
    "optim": "tts002",
    "max_text_chars": 0,
}
# Leave-one-out ablation (docs/ABLATION_TTS_0.10.md). Keep this a plain literal:
# tools/build_kaggle_full_train.py reads it to export one notebook per arm.
ARMS = {
    "R0": {"revert": [], "seed": 2026, "title": "complete TTS_0.10"},
    "R0b": {"revert": [], "seed": 2027, "title": "complete TTS_0.10, second seed"},
    "E0": {"revert": ["text_encoder", "posterior_encoder", "decoder", "discriminator",
                      "feature_matching", "recon_loss", "optim", "max_text_chars"],
           "seed": 2026, "title": "every switch at TTS_0.02"},
    "G1": {"revert": ["text_encoder"], "seed": 2026, "title": "TTS_0.02 text encoder"},
    "G2": {"revert": ["posterior_encoder"], "seed": 2026, "title": "TTS_0.02 posterior encoder"},
    "G3": {"revert": ["decoder"], "seed": 2026, "title": "TTS_0.02 decoder"},
    "A1": {"revert": ["discriminator"], "seed": 2026, "title": "TTS_0.02 discriminator"},
    "A2": {"revert": ["feature_matching"], "seed": 2026, "title": "no feature matching"},
    "L1": {"revert": ["recon_loss"], "seed": 2026, "title": "TTS_0.02 reconstruction loss"},
    "O1": {"revert": ["optim"], "seed": 2026, "title": "TTS_0.02 optimisation"},
    "D1": {"revert": ["max_text_chars"], "seed": 2026, "title": "no text-length filter"},
    # TTS_0.11 base (docs/TTS_0.11.md): D1's recipe, the only change the ablation found beneficial,
    # with the long-sentence test set also held out so long sentences can be evaluated fairly.
    "T011": {"revert": ["max_text_chars"], "seed": 2026,
             "title": "TTS_0.11 base: no text-length filter, long test sentences held out",
             "heldout_extra": ["ljspeech_long_100.txt"]},
    # TTS_0.20 candidates (docs/TTS_0.20.md): T011 plus options from TTS020_OPTIONS.
    "V1": {"revert": ["max_text_chars"], "seed": 2026, "heldout_extra": ["ljspeech_long_100.txt"],
           "options": {"encoder_window": [12, 4]},
           "title": "T011 + windowed text encoder (12 words back, 0-4 ahead)"},
    "V2": {"revert": ["max_text_chars"], "seed": 2026, "heldout_extra": ["ljspeech_long_100.txt"],
           "options": {"text_input": "phonemes"},
           "title": "T011 + phoneme input (g2p_en)"},
    "V3": {"revert": ["max_text_chars"], "seed": 2026, "heldout_extra": ["ljspeech_long_100.txt"],
           "options": {"text_input": "phonemes", "encoder_window": [12, 4]},
           "title": "T011 + phoneme input + windowed text encoder"},
    "V4": {"revert": ["max_text_chars"], "seed": 2026, "heldout_extra": ["ljspeech_long_100.txt"],
           "options": {"duration_predictor": "stochastic"},
           "title": "T011 + stochastic duration predictor"},
}
# TTS_0.20 options, off by default. Only options that differ from these defaults are added to
# an arm's switches, so every TTS_0.10 arm keeps exactly its recorded switches.
#   text_input          "chars" | "phonemes" (g2p_en, sts_eval/phonemes/, tools/phonemize.py)
#   encoder_window      None | [left_words, max_lookahead_words]: each token attends to that many
#                       words back and 0..max_lookahead words ahead (drawn per training batch);
#                       the encoder's convolutions become causal, so nothing leaks beyond it
#   duration_predictor  "deterministic" | "stochastic" (the flow-based predictor of VITS)
TTS020_OPTIONS = {"text_input": "chars", "encoder_window": None, "duration_predictor": "deterministic"}
OPTIM_PRESETS = {
    "tts010": {"batch_size": 16, "learning_rate": 2e-4, "amp": True, "max_grad_norm": None},
    "tts002": {"batch_size": 8, "learning_rate": 1e-4, "amp": False, "max_grad_norm": 1.0},
}

ARM = os.environ.get("STS_ARM", "R0")
if ARM not in ARMS:
    raise ValueError(f"Unknown STS_ARM {ARM!r}; choose one of {sorted(ARMS)}.")
SWITCHES = {name: (TTS002_SWITCHES if name in ARMS[ARM]["revert"] else TTS010_SWITCHES)[name]
            for name in TTS010_SWITCHES}
unknown_options = set(ARMS[ARM].get("options", {})) - set(TTS020_OPTIONS)
if unknown_options:
    raise ValueError(f"Unknown TTS_0.20 options {sorted(unknown_options)} for arm {ARM}.")
SWITCHES.update({name: value for name, value in ARMS[ARM].get("options", {}).items()
                 if value != TTS020_OPTIONS[name]})
TRAIN_SEED = ARMS[ARM]["seed"]
OPTIM = OPTIM_PRESETS[SWITCHES["optim"]]

# Seed before anything draws random numbers (demo batch, model initialisation, data order).
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)
# Deterministic cuDNN kernels made a TTS_0.10 step 3.2x slower (0.78 s vs 0.25 s, RTX 4070 Ti
# SUPER). Runs are seeded but, as in benlu's trainer, not bitwise reproducible.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False

# Repository sources: shared evaluator and the held-out split.
REPO_ROOT = Path(os.environ.get("STS_REPO_ROOT", ".." if Path.cwd().name == "notebooks" else ".")).resolve()
if not (REPO_ROOT / "sts_eval" / "reconstruction.py").is_file():
    raise FileNotFoundError("Set STS_REPO_ROOT to the STS2026 repository containing sts_eval/.")
HELDOUT_PATH = REPO_ROOT / "sts_eval" / "splits" / "ljspeech_heldout_100.txt"
# Further test sets an arm also keeps out of training (T011: the long-sentence set).
HELDOUT_EXTRA_PATHS = [REPO_ROOT / "sts_eval" / "splits" / name for name in ARMS[ARM].get("heldout_extra", [])]


def find_ljspeech():
    """Use STS_DATA_DIR, or the first attached Kaggle dataset containing metadata.csv and wavs/."""
    override = os.environ.get("STS_DATA_DIR")
    if override:
        return Path(override)
    for candidate in sorted(Path("/kaggle/input").glob("**/metadata.csv")):
        if (candidate.parent / "wavs").is_dir():
            return candidate.parent
    raise FileNotFoundError("Attach the LJSpeech-1.1 dataset (metadata.csv + wavs/) or set STS_DATA_DIR.")


DATA_DIR = find_ljspeech()
WAV_DIR = DATA_DIR / "wavs"
SPEAKER_MANIFEST = os.environ.get("STS_SPEAKER_MANIFEST") or None
METADATA_PATH = Path(SPEAKER_MANIFEST) if SPEAKER_MANIFEST else DATA_DIR / "metadata.csv"

# Audio representation
SAMPLE_RATE = 22_050
N_FFT = 1_024
HOP_LENGTH = 256
WIN_LENGTH = 1_024
N_MELS = 80
SEGMENT_SAMPLES = 8_192
SEGMENT_FRAMES = SEGMENT_SAMPLES // HOP_LENGTH

BATCH_SIZE = OPTIM["batch_size"]
NUM_WORKERS = int(os.environ.get("STS_NUM_WORKERS", "4"))
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
USE_AMP = OPTIM["amp"] and DEVICE.type == "cuda"

print(f"{MODEL_VERSION} arm {ARM} ({ARMS[ARM]['title']}), seed {TRAIN_SEED}")
print("Switches:", SWITCHES)
print("Optimisation:", OPTIM, "| autocast:", USE_AMP)
print("Dataset:", DATA_DIR)
print(f"PyTorch {torch.__version__}; device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(DEVICE)})" if DEVICE.type == "cuda" else ""))

## 2. Metadata, held-out split and length filter

LJSpeech rows contain a file ID, the original transcription and a normalized transcription.
`csv.QUOTE_NONE` keeps quotation marks as text; without it 16 transcriptions were lost.

The 100 clips in `sts_eval/splits/ljspeech_heldout_100.txt` are removed before training in
every arm. With `max_text_chars = 100`, only clips whose normalized text has 1–100 characters
remain (6,208 clips); without the filter 13,000 clips remain.

In [ ]:
def load_training_metadata(path, conditioned=False):
    """Read one corpus and assign stable, lexically sorted speaker IDs."""
    read = functools.partial(pd.read_csv, sep="|", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    if conditioned:
        frame = read(path)
        required = {"file_name", "normalized_transcription", "speaker_name"}
        if not required.issubset(frame.columns):
            raise ValueError(f"Speaker manifest needs columns: {sorted(required)}")
        frame = frame[["file_name", "normalized_transcription", "speaker_name"]].copy()
    else:
        frame = read(path, header=None, names=["file_name", "transcription", "normalized_transcription"])
        frame = frame[["file_name", "normalized_transcription"]].copy()
    for column in frame.columns:
        frame[column] = frame[column].str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"Empty {column} in {path}.")
    if conditioned:
        names = sorted(frame["speaker_name"].unique())
        speaker_to_id = {name: index for index, name in enumerate(names)}
        frame["speaker_id"] = frame["speaker_name"].map(speaker_to_id)
    else:
        speaker_to_id = {}
    frame = frame.reset_index(drop=True)
    if frame.empty or frame["file_name"].duplicated().any():
        raise ValueError("Metadata must contain unique, nonempty file IDs.")
    return frame, speaker_to_id


def load_id_list(path):
    """Read one ID per line, ignoring blank lines and # comments."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    return [line.strip() for line in lines if line.strip() and not line.startswith("#")]


def heldout_ids(base_path, extra_paths=()):
    """IDs never trained on: the base split, then each extra test set (order kept, no duplicates)."""
    ids = load_id_list(base_path)
    for path in extra_paths:
        ids += [name for name in load_id_list(path) if name not in set(ids)]
    return ids


def apply_phoneme_cache(frame, cache_path, metadata_path):
    """Replace normalized_transcription by its phonemes (tools/phonemize.py); keep the text in "text"."""
    import hashlib
    header, phonemes = {}, {}
    for line in Path(cache_path).read_text(encoding="utf-8").splitlines():
        if line.startswith("# "):
            key, _, value = line[2:].partition(": ")
            header[key] = value
        elif line:
            name, _, value = line.partition("|")
            phonemes[name] = value
    if header.get("format") != "sts2026-phonemes-g2p_en-v1":
        raise ValueError(f"{cache_path} is not a phoneme cache written by tools/phonemize.py.")
    digest = hashlib.sha256(Path(metadata_path).read_bytes()).hexdigest()
    if header.get("metadata_sha256") != digest:
        raise ValueError(f"{cache_path} was made from another metadata.csv.")
    missing = [name for name in frame["file_name"] if name not in phonemes]
    if missing:
        raise ValueError(f"The phoneme cache lacks {len(missing)} clips, e.g. {missing[:3]}.")
    frame = frame.copy()
    frame["text"] = frame["normalized_transcription"]
    frame["normalized_transcription"] = frame["file_name"].map(phonemes)
    words_differ = frame["text"].str.split().str.len() != frame["normalized_transcription"].str.split(" ").str.len()
    if words_differ.any():
        raise ValueError(f"Phoneme word counts differ for {frame.loc[words_differ, 'file_name'].tolist()[:3]}.")
    return frame


def filter_text_length(frame, max_chars):
    """Keep normalized transcriptions of 1..max_chars characters; 0 disables the filter."""
    if not max_chars:
        return frame.reset_index(drop=True)
    lengths = frame["normalized_transcription"].str.len()
    return frame[(lengths >= 1) & (lengths <= max_chars)].reset_index(drop=True)


corpus_df, SPEAKER_TO_ID = load_training_metadata(METADATA_PATH, bool(SPEAKER_MANIFEST))
TEXT_INPUT = SWITCHES.get("text_input", "chars")
PHONEME_CACHE = None
if TEXT_INPUT == "phonemes":
    PHONEME_CACHE = Path(os.environ.get("STS_PHONEME_CACHE")
                         or REPO_ROOT / "sts_eval" / "phonemes" / "ljspeech_g2p_en.tsv")
    corpus_df = apply_phoneme_cache(corpus_df, PHONEME_CACHE, METADATA_PATH)
    print(f"Phoneme input from {PHONEME_CACHE}")
NUM_SPEAKERS = len(SPEAKER_TO_ID)
GIN_CHANNELS = int(os.environ.get("STS_GIN_CHANNELS", "64")) if NUM_SPEAKERS else 0
if NUM_SPEAKERS and GIN_CHANNELS <= 0:
    raise ValueError("STS_GIN_CHANNELS must be positive in speaker mode.")
ALLOW_SUBSET = os.environ.get("STS_ALLOW_SUBSET") == "1"

# The held-out split belongs to LJSpeech; a speaker manifest has no held-out split.
HELDOUT_BASE_IDS = [] if SPEAKER_MANIFEST else load_id_list(HELDOUT_PATH)
HELDOUT_IDS = [] if SPEAKER_MANIFEST else heldout_ids(HELDOUT_PATH, HELDOUT_EXTRA_PATHS)
heldout_df = corpus_df[corpus_df["file_name"].isin(set(HELDOUT_IDS))].reset_index(drop=True)
if len(heldout_df) != len(HELDOUT_IDS) and not ALLOW_SUBSET:
    raise ValueError("metadata.csv lacks held-out IDs; use the complete LJSpeech-1.1 metadata.")
metadata_df = corpus_df[~corpus_df["file_name"].isin(set(HELDOUT_IDS))]
metadata_df = filter_text_length(metadata_df, SWITCHES["max_text_chars"])

# Never silently drop missing recordings outside a smoke test.
missing = [name for name in metadata_df["file_name"] if not (WAV_DIR / f"{name}.wav").is_file()]
if missing:
    if SPEAKER_MANIFEST or not ALLOW_SUBSET:
        raise FileNotFoundError(f"{len(missing)} of {len(metadata_df)} wav files are missing under {WAV_DIR}.")
    metadata_df = metadata_df[~metadata_df["file_name"].isin(missing)].reset_index(drop=True)
    print(f"SMOKE TEST ONLY: training on {len(metadata_df)} available clips.")
present = [(WAV_DIR / f"{name}.wav").is_file() for name in heldout_df["file_name"]]
heldout_df = heldout_df[pd.Series(present, index=heldout_df.index, dtype=bool)].reset_index(drop=True)
print(f"Corpus clips: {len(corpus_df)}; held out: {len(HELDOUT_IDS)} "
      f"({len(heldout_df)} recordings found); training clips: {len(metadata_df)}; "
      f"max_text_chars: {SWITCHES['max_text_chars'] or 'none'}; speakers: {SPEAKER_TO_ID}")
metadata_df.head()

## 3. Convert text into token IDs

Every supported character is assigned an integer ID. ID `0` is reserved for padding, which
allows text sequences of different lengths to share one batch. Unsupported characters are
ignored rather than mapped to an unknown token. The 76-symbol inventory is identical in
TTS_0.02 and benlu's notebook.

In [ ]:
PAD_SYMBOL = "_"
PUNCTUATION = ";:,.!?¡¿—…\"«»“”ü ()-'[]"
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

SYMBOLS = [PAD_SYMBOL] + list(PUNCTUATION) + list(LETTERS)

symbol_to_id = {
    symbol: index
    for index, symbol in enumerate(SYMBOLS)
}

PAD_ID = symbol_to_id[PAD_SYMBOL]


def token_table(text_input="chars"):
    """Token IDs of the text input: characters (TTS_0.10) or phonemes (tools/phonemize.py)."""
    if text_input == "chars":
        symbols = (["_"] + list(";:,.!?¡¿—…\"«»“”ü ()-'[]")
                   + list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"))
    elif text_input == "phonemes":
        # one character per ARPAbet phoneme: chr(0x100 + index) in tools/phonemize.py ARPABET
        symbols = ["_", " "] + list(",.?!-'") + [chr(0x100 + index) for index in range(70)]
    else:
        raise ValueError(f"Unknown text input {text_input!r}.")
    return {symbol: index for index, symbol in enumerate(symbols)}


# The arm's text input decides the token table (characters for every TTS_0.10 arm).
symbol_to_id = token_table(globals().get("SWITCHES", {}).get("text_input", "chars"))
PAD_ID = symbol_to_id[PAD_SYMBOL]


def encode_text(text, table=None):
    """
    Convert a string into a sequence of integer token IDs.

    Example:
        "Hi" -> tensor([token_id_for_H, token_id_for_i])
    """
    table = symbol_to_id if table is None else table
    token_ids = [
        table[character]
        for character in text
        if character in table
    ]

    return torch.tensor(token_ids, dtype=torch.long)

## 4. Audio preprocessing and dataset

The waveform is converted into a magnitude spectrogram with the Short-Time Fourier Transform.
With `N_FFT = 1024` it has 513 frequency bins. TTS_0.10 uses the VITS magnitude
`sqrt(|X|² + 1e-6)` (TTS_0.02 used `|X|`), so benlu's trained weights see the input they were
trained on.

The `tts010` reconstruction loss compares Slaney-scale log-mel spectrograms, the librosa
default used by VITS. The filterbank is built with torchaudio, so librosa is not required.

`LJSpeechDataset` returns encoded text, spectrogram and waveform for each utterance. It
stores every setting it needs, so DataLoader worker processes also work on Windows.

In [ ]:
def waveform_to_spectrogram(waveform, n_fft=1024, hop_length=256, win_length=1024):
    """
    Convert [..., audio_length] audio into a [..., n_fft // 2 + 1, frames] magnitude
    spectrogram, sqrt(|X|^2 + 1e-6), with (n_fft - hop) / 2 reflect padding and no centring.
    A [1, audio_length] waveform gives [frequency_bins, time_frames].
    """
    padding = (n_fft - hop_length) // 2
    squeeze = waveform.dim() == 1
    waveform = waveform.unsqueeze(0) if squeeze else waveform
    waveform = F.pad(waveform.unsqueeze(1), (padding, padding), mode="reflect").squeeze(1)
    window = torch.hann_window(win_length, device=waveform.device, dtype=waveform.dtype)
    spectrum = torch.stft(
        waveform,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=False,
        normalized=False,
        onesided=True,
        return_complex=True,
    )
    magnitude = torch.sqrt(spectrum.real.pow(2) + spectrum.imag.pow(2) + 1e-6)
    return magnitude.squeeze(0)


@functools.lru_cache(maxsize=None)
def slaney_mel_basis(n_mels=80, n_fft=1024, sample_rate=22050, f_min=0.0, f_max=None,
                     device="cpu", dtype=torch.float32):
    """[n_mels, n_fft // 2 + 1] Slaney filterbank (librosa.filters.mel defaults); cached, do not modify."""
    basis = torchaudio.functional.melscale_fbanks(
        n_fft // 2 + 1, f_min, f_max or sample_rate / 2, n_mels, sample_rate,
        norm="slaney", mel_scale="slaney",
    )
    return basis.T.contiguous().to(device=device, dtype=dtype)


def linear_to_log_mel(spectrogram, n_mels=80, n_fft=1024, sample_rate=22050):
    """[B, n_fft // 2 + 1, T] magnitude -> [B, n_mels, T] natural-log mel, floor 1e-5."""
    basis = slaney_mel_basis(n_mels, n_fft, sample_rate, device=spectrogram.device, dtype=spectrogram.dtype)
    return torch.log(torch.clamp(torch.matmul(basis, spectrogram), min=1e-5))


def waveform_to_log_mel(waveform, n_mels=80, n_fft=1024, hop_length=256, win_length=1024, sample_rate=22050):
    """[B, samples] audio -> [B, n_mels, frames] with the training spectrogram framing."""
    spectrogram = waveform_to_spectrogram(waveform, n_fft, hop_length, win_length)
    return linear_to_log_mel(spectrogram, n_mels, n_fft, sample_rate)

In [ ]:
def word_spans(tokens, space_id):
    """[(start, end)] token ranges of the words: maximal runs of non-space tokens."""
    spans, start = [], None
    for index, token in enumerate(list(tokens.tolist()) + [space_id]):
        if token != space_id and start is None:
            start = index
        elif token == space_id and start is not None:
            spans.append((start, index))
            start = None
    return spans


def word_crop_bounds(tokens, durations, space_id, rng, min_words=1, max_words=8, prefix_share=0.5,
                     min_frames=32, boundary_ids=None):
    """
    Choose whole words to keep for chunk-aware training.

    durations[i] is the number of spectrogram frames of token i (MAS of a trained model). The
    crop is a prefix (probability prefix_share) or a span starting at a random word, of
    min_words..max_words words, extended by whole words until it covers min_frames frames.
    Spaces around the kept words are dropped. Returns (token_start, token_end, frame_start,
    frame_end), or None for a text with fewer than two words.

    boundary_ids (TTS_0.20 option): the crop ends where a word ends in one of these tokens
    (punctuation) or at the end of the text, the nearest such end within min_words..max_words
    words when there is one, and is extended to the next such end to reach min_frames.
    """
    words = word_spans(tokens, space_id)
    if len(words) < 2:
        return None
    bounds = np.concatenate([[0], np.cumsum(durations)])
    count = rng.randint(min(min_words, len(words)), min(max_words, len(words)))
    first = 0 if rng.random() < prefix_share else rng.randint(0, len(words) - count)
    last = first + count  # exclusive word index
    if boundary_ids:
        ends = [end for end in range(1, len(words) + 1)
                if end == len(words) or int(tokens[words[end - 1][1] - 1]) in boundary_ids]
        in_range = [end for end in ends if first + min_words <= end <= first + max_words]
        if in_range:
            last = in_range[0]
        while (bounds[words[last - 1][1]] - bounds[words[first][0]] < min_frames
               and (first > 0 or last < len(words))):
            later = [end for end in ends if end > last]
            if later:
                last = later[0]
            else:
                first -= 1
    while (bounds[words[last - 1][1]] - bounds[words[first][0]] < min_frames
           and (first > 0 or last < len(words))):
        if last < len(words):
            last += 1
        else:
            first -= 1
    token_start, token_end = words[first][0], words[last - 1][1]
    return token_start, token_end, int(bounds[token_start]), int(bounds[token_end])


def load_chunk_durations(path, hop_length, expected_format="sts2026-chunk-durations-v1"):
    """Per-token durations written by tools/chunk_alignments.py, and their provenance."""
    import hashlib
    import json
    path = Path(path)
    raw = path.read_bytes()
    record = json.loads(raw)
    if record.get("format") != expected_format or record.get("hop_length") != hop_length:
        raise ValueError(f"{path} is not a {expected_format} file for hop length {hop_length}.")
    provenance = {"path": str(path.resolve()), "sha256": hashlib.sha256(raw).hexdigest(),
                  "clips": len(record["durations"]),
                  **{key: record.get(key) for key in ("source_checkpoint", "source_checkpoint_sha256",
                                                      "source_step", "alignment")}}
    return record["durations"], provenance


class LJSpeechDataset(Dataset):
    def __init__(self, metadata, wav_dir, sample_rate=22050, n_fft=1024, hop_length=256,
                 win_length=1024, table=None, chunk_training=None, chunk_durations=None):
        self.metadata = metadata.reset_index(drop=True)
        self.wav_dir = Path(wav_dir)
        self.sample_rate = sample_rate
        self.stft = (n_fft, hop_length, win_length)
        # Keep the token table on the instance so spawned workers need no notebook globals.
        self.table = dict(symbol_to_id if table is None else table)
        # Chunk-aware training (section 4b); None keeps every clip whole.
        self.chunk_training = dict(chunk_training) if chunk_training else None
        self.chunk_durations = None
        if self.chunk_training:
            names = self.metadata["file_name"].tolist()
            missing = [name for name in names if name not in chunk_durations]
            if missing:
                raise ValueError(f"Chunk durations lack {len(missing)} clips, e.g. {missing[:3]}.")
            self.chunk_durations = {name: list(chunk_durations[name]) for name in names}
            for name, text in zip(names, self.metadata["normalized_transcription"]):
                if len(self.chunk_durations[name]) != encode_text(text, self.table).numel():
                    raise ValueError(f"Chunk durations of {name} do not match its text.")

    def __len__(self):
        return len(self.metadata)

    def _chunk_crop(self, file_name, text, waveform):
        """Crop text and waveform to the same whole words (frames from the cached durations)."""
        n_fft, hop_length, _ = self.stft
        settings, durations = self.chunk_training, self.chunk_durations[file_name]
        frames = (waveform.size(-1) + 2 * ((n_fft - hop_length) // 2) - n_fft) // hop_length + 1
        if sum(durations) != frames:
            raise ValueError(f"Chunk durations of {file_name} cover {sum(durations)} frames, audio has {frames}.")
        bounds = word_crop_bounds(text, durations, settings["space_id"], random, settings["min_words"],
                                  settings["max_words"], settings["prefix_share"], settings["min_frames"],
                                  settings.get("boundary_ids"))
        if bounds is None:
            return text, waveform
        token_start, token_end, frame_start, frame_end = bounds
        return text[token_start:token_end], waveform[:, frame_start * hop_length:frame_end * hop_length]

    def __getitem__(self, index):
        row = self.metadata.iloc[index]

        # 1. Load and encode the text
        text = encode_text(row["normalized_transcription"], self.table)

        # 2. Load the waveform
        audio_path = self.wav_dir / f"{row['file_name']}.wav"
        waveform, sample_rate = torchaudio.load(audio_path)

        if sample_rate != self.sample_rate:
            raise ValueError(
                f"Expected sample rate {self.sample_rate}, "
                f"but received {sample_rate} for {audio_path.name}"
            )

        # LJSpeech is mono, so waveform has shape [1, audio_length]
        if waveform.size(0) != 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Chunk-aware training: with probability prob, keep only some whole words.
        if self.chunk_training is not None and random.random() < self.chunk_training["prob"]:
            text, waveform = self._chunk_crop(row["file_name"], text, waveform)

        # 3. Convert the waveform into a spectrogram
        spectrogram = waveform_to_spectrogram(waveform, *self.stft)

        sample = (text, spectrogram, waveform)
        if "speaker_id" in self.metadata.columns:
            return (*sample, int(row["speaker_id"]))
        return sample

### Padding variable-length samples

Utterances and transcriptions have different lengths. The collate function records every original length, pads each batch to its longest item, and returns the lengths so later masks can distinguish real values from padding.

In [ ]:
def collate_tts_batch(batch, pad_id=None):
    """
    Combine samples of different lengths into one padded batch.
    """
    pad_id = PAD_ID if pad_id is None else pad_id
    if not batch or len({len(sample) for sample in batch}) != 1 or len(batch[0]) not in (3, 4):
        raise ValueError("Batch must contain uniformly unconditioned or speaker-ID samples.")
    texts, spectrograms, waveforms = zip(*(sample[:3] for sample in batch))
    speaker_ids = None
    if len(batch[0]) == 4:
        ids = [sample[3] for sample in batch]
        if any(isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 0 for value in ids):
            raise ValueError("Speaker IDs must be nonnegative integers.")
        speaker_ids = torch.tensor(ids, dtype=torch.long)

    # Record the original lengths before padding
    text_lengths = torch.tensor(
        [text.size(0) for text in texts],
        dtype=torch.long
    )

    spec_lengths = torch.tensor(
        [spectrogram.size(1) for spectrogram in spectrograms],
        dtype=torch.long
    )

    wav_lengths = torch.tensor(
        [waveform.size(1) for waveform in waveforms],
        dtype=torch.long
    )

    # Text:
    # List of [text_length] -> [batch, max_text_length]
    text_padded = pad_sequence(
        texts,
        batch_first=True,
        padding_value=pad_id
    )

    # Spectrogram:
    # [frequency, time] -> [time, frequency]
    # so pad_sequence can pad the time dimension
    spec_padded = pad_sequence(
        [
            spectrogram.transpose(0, 1)
            for spectrogram in spectrograms
        ],
        batch_first=True,
        padding_value=0.0
    )

    # [batch, time, frequency]
    # -> [batch, frequency, time]
    spec_padded = spec_padded.transpose(1, 2)

    # Waveform:
    # [1, time] -> [time]
    wav_padded = pad_sequence(
        [
            waveform.squeeze(0)
            for waveform in waveforms
        ],
        batch_first=True,
        padding_value=0.0
    )

    # [batch, time] -> [batch, 1, time]
    wav_padded = wav_padded.unsqueeze(1)

    tensors = (
        text_padded,
        text_lengths,
        spec_padded,
        spec_lengths,
        wav_padded,
        wav_lengths
    )
    return tensors if speaker_ids is None else (*tensors, speaker_ids)

### 4b. Chunk-aware training (optional, off by default)

A voice agent can start speaking before the language model finishes the sentence by
synthesizing the text in chunks of a few words. Chunk-aware training shows the model such
pieces: with probability `STS_CHUNK_PROB` (default 0 = off) each training clip is cut to whole
words together with the matching audio. The kept words are a prefix of the sentence (share
`STS_CHUNK_PREFIX_SHARE`, default 0.5) or a span starting at a random word, `STS_CHUNK_WORDS`
words long (default `1-8`), extended by whole words to at least one decoder segment (32 frames,
0.37 s). Spaces around the kept words are dropped.

The audio cut points come from per-character durations computed once by the MAS of a trained
checkpoint (`tools/chunk_alignments.py`, posterior mean, held-out clips skipped) and read from
`STS_CHUNK_DURATIONS`. During training MAS still aligns each cropped clip itself. With the
option off, the dataset draws no random numbers and every clip is used whole, as before.

In [ ]:
# Chunk-aware training (section 4b): off unless STS_CHUNK_PROB > 0.
CHUNK_PROB = float(os.environ.get("STS_CHUNK_PROB") or 0)
CHUNK_TRAINING, chunk_durations = None, None
if CHUNK_PROB > 0:
    if not os.environ.get("STS_CHUNK_DURATIONS"):
        raise ValueError("STS_CHUNK_PROB > 0 needs STS_CHUNK_DURATIONS (written by tools/chunk_alignments.py).")
    low, high = (int(value) for value in os.environ.get("STS_CHUNK_WORDS", "1-8").split("-"))
    CHUNK_TRAINING = {
        "prob": CHUNK_PROB, "min_words": low, "max_words": high,
        "prefix_share": float(os.environ.get("STS_CHUNK_PREFIX_SHARE", "0.5")),
        "min_frames": SEGMENT_FRAMES, "space_id": symbol_to_id[" "],
    }
    # TTS_0.20 option: end crops at punctuation instead of after any word.
    CHUNK_BOUNDARIES = os.environ.get("STS_CHUNK_BOUNDARIES", "words")
    if CHUNK_BOUNDARIES == "punctuation":
        CHUNK_TRAINING["boundaries"] = "punctuation"
        CHUNK_TRAINING["boundary_ids"] = sorted(symbol_to_id[c] for c in ",.?!;:" if c in symbol_to_id)
    elif CHUNK_BOUNDARIES != "words":
        raise ValueError("STS_CHUNK_BOUNDARIES must be words or punctuation.")
    if not (CHUNK_PROB <= 1 and 1 <= low <= high and 0 <= CHUNK_TRAINING["prefix_share"] <= 1):
        raise ValueError(f"Invalid chunk-training settings: {CHUNK_TRAINING}")
    chunk_durations, CHUNK_TRAINING["durations"] = load_chunk_durations(
        os.environ["STS_CHUNK_DURATIONS"], HOP_LENGTH)
    print("Chunk-aware training:", CHUNK_TRAINING)

train_dataset = LJSpeechDataset(
    metadata=metadata_df,
    wav_dir=WAV_DIR,
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    chunk_training=CHUNK_TRAINING,
    chunk_durations=chunk_durations,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=functools.partial(collate_tts_batch, pad_id=PAD_ID),
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

print(f"{len(train_dataset)} training clips, batch size {BATCH_SIZE}, "
      f"{len(train_loader)} batches per pass over the data")

## 5. Sequence masks

A mask marks valid positions with `True` and padded positions with `False`. Every encoder and loss uses these masks so padding does not affect representations, alignments, or loss values.

In [ ]:
def create_sequence_mask(lengths, max_length):
    """
    Create a mask for sequences with different lengths.

    lengths:
        [3, 5]

    result:
        [[True, True, True, False, False],
         [True, True, True, True,  True ]]
    """
    positions = torch.arange(
        max_length,
        device=lengths.device
    )

    return positions.unsqueeze(0) < lengths.unsqueeze(1)

## 6. Text encoder and latent prior

The text encoder combines character and position embeddings, then applies Transformer layers. For every text token it returns:

- contextual text features for the duration predictor;
- a prior mean $\mu_p$;
- a prior log standard deviation $\log \sigma_p$;
- a text mask.

These statistics describe a **text-conditioned Gaussian prior**. MAS later expands the token-level prior to spectrogram-frame length.

**`text_encoder = tts010`** (`TextEncoder`, benlu): token embeddings are initialised with
standard deviation $d^{-1/2}$ and scaled by $\sqrt{d}$; each block is post-norm self-attention
followed by a feed-forward network made of two kernel-3 convolutions with ReLU, so every token
also sees its neighbours. Padding is zeroed after every block.

**`text_encoder = tts002`** (`LegacyTextEncoder`): PyTorch `nn.TransformerEncoder` with a
GELU linear feed-forward network.

In [ ]:
class ConvTransformerBlock(nn.Module):
    def __init__(
        self,
        hidden_channels=192,
        feedforward_channels=448,
        num_heads=2,
        kernel_size=3,
        dropout=0.1,
        causal_ffn=False,
    ):
        super().__init__()
        if kernel_size % 2 == 0:
            raise ValueError("kernel_size must be odd")
        # TTS_0.20 windowed encoder: the feed-forward convolutions see only the past.
        self.causal_ffn = causal_ffn
        self.ffn_left_pad = kernel_size - 1 if causal_ffn else 0

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.conv1 = nn.Conv1d(
            hidden_channels,
            feedforward_channels,
            kernel_size,
            padding=0 if causal_ffn else kernel_size // 2,
        )
        self.conv2 = nn.Conv1d(
            feedforward_channels,
            hidden_channels,
            kernel_size,
            padding=0 if causal_ffn else kernel_size // 2,
        )

        self.activation = nn.ReLU()
        self.ffn_dropout = nn.Dropout(dropout)
        self.attention_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(hidden_channels)

    def forward(self, x, valid_mask, attn_mask=None):
        # x: [B, T, H]
        # valid_mask: [B, T], True = valid token
        # attn_mask: optional [B * heads, T, T], True = may not attend (windowed encoder)
        padding_mask = ~valid_mask
        padding_bth = padding_mask.unsqueeze(-1)

        x = x.masked_fill(padding_bth, 0.0)

        attention_output, _ = self.attention(
            x, x, x,
            key_padding_mask=padding_mask,
            attn_mask=attn_mask,
            need_weights=False,
        )

        x = self.norm1(
            x + self.attention_dropout(attention_output)
        )
        x = x.masked_fill(padding_bth, 0.0)

        # Convolutional FFN: [B, H, T]
        padding_bct = padding_mask.unsqueeze(1)
        y = x.transpose(1, 2)

        y = self.conv1(F.pad(y, (self.ffn_left_pad, 0)))
        y = self.activation(y)
        y = self.ffn_dropout(y)

        # Remove padding activations before the next convolution.
        y = y.masked_fill(padding_bct, 0.0)
        y = self.conv2(F.pad(y, (self.ffn_left_pad, 0)))
        y = y.masked_fill(padding_bct, 0.0)

        y = y.transpose(1, 2)

        x = self.norm2(x + self.output_dropout(y))
        return x.masked_fill(padding_bth, 0.0)


def window_attention_mask(token_ids, space_id, left_words, lookahead_words, num_heads):
    """
    [B * heads, T, T] boolean mask, True = blocked: token i may attend to token j only if j's word
    is at most left_words before and lookahead_words after i's word. A space belongs to the word
    after it, so a token never sees the text beyond its own lookahead.
    """
    word = torch.cumsum((token_ids == space_id).long(), dim=1)
    offset = word.unsqueeze(1) - word.unsqueeze(2)  # [B, query i, key j]: word(j) - word(i)
    blocked = (offset > lookahead_words) | (offset < -left_words)
    return blocked.repeat_interleave(num_heads, dim=0)


class TextEncoder(nn.Module):
    """TTS_0.10 text encoder (benlu)."""

    def __init__(
        self,
        vocab_size,
        hidden_channels=192,
        latent_channels=192,
        feedforward_channels=768,
        num_heads=2,
        num_layers=6,
        dropout=0.1,
        max_text_length=512,
        padding_id=0,
        kernel_size=3,
        window=None,
        space_id=None,
    ):
        super().__init__()

        self.hidden_channels = hidden_channels
        self.max_text_length = max_text_length
        self.num_heads = num_heads
        # TTS_0.20 windowed encoder: [left_words, max_lookahead_words], or None (full attention).
        self.window = None if window is None else (int(window[0]), int(window[1]))
        if self.window is not None and space_id is None:
            raise ValueError("A windowed text encoder needs the space token ID.")
        self.space_id = space_id
        # Lookahead used outside training; chunked synthesis sets it to its own lookahead.
        self.lookahead_words = None if self.window is None else self.window[1]

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_channels,
            padding_idx=padding_id,
        )
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=hidden_channels ** -0.5,
        )

        if padding_id is not None:
            with torch.no_grad():
                self.token_embedding.weight[padding_id].zero_()

        self.position_embedding = nn.Embedding(
            max_text_length,
            hidden_channels,
        )
        nn.init.normal_(
            self.position_embedding.weight,
            mean=0.0,
            std=hidden_channels ** -0.5,
        )

        # Each block is initialized independently.
        self.layers = nn.ModuleList([
            ConvTransformerBlock(
                hidden_channels=hidden_channels,
                feedforward_channels=feedforward_channels,
                num_heads=num_heads,
                kernel_size=kernel_size,
                dropout=dropout,
                causal_ffn=self.window is not None,
            )
            for _ in range(num_layers)
        ])

        self.statistics_projection = nn.Linear(
            hidden_channels,
            latent_channels * 2,
        )

    def forward(self, token_ids, text_lengths):
        # token_ids: [B, T], right-padded
        # text_lengths: [B]
        batch_size, text_length = token_ids.shape

        if text_length > self.max_text_length:
            raise ValueError(
                f"Text length {text_length} exceeds "
                f"max_text_length={self.max_text_length}"
            )

        text_lengths = text_lengths.to(token_ids.device)

        if text_lengths.shape != (batch_size,):
            raise ValueError("text_lengths must have shape [batch]")

        if ((text_lengths < 1) | (text_lengths > text_length)).any():
            raise ValueError("Each text length must be in [1, T]")

        positions = torch.arange(
            text_length,
            device=token_ids.device,
        )

        valid_mask = (
            positions.unsqueeze(0) < text_lengths.unsqueeze(1)
        )
        padding_bth = ~valid_mask.unsqueeze(-1)

        x = self.token_embedding(token_ids)
        x = x * math.sqrt(self.hidden_channels)
        x = x + self.position_embedding(positions).unsqueeze(0)
        x = x.masked_fill(padding_bth, 0.0)

        # Windowed encoder: only the first layer looks ahead (K words); later layers look back only,
        # so stacking layers never widens the lookahead and a token's output is final once the
        # text K words beyond its word has arrived.
        masks = [None] * len(self.layers)
        if self.window is not None:
            lookahead = (int(torch.randint(0, self.window[1] + 1, ())) if self.training
                         else self.lookahead_words)
            first = window_attention_mask(token_ids, self.space_id, self.window[0], lookahead, self.num_heads)
            rest = window_attention_mask(token_ids, self.space_id, self.window[0], 0, self.num_heads)
            masks = [first] + [rest] * (len(self.layers) - 1)

        for layer, attn_mask in zip(self.layers, masks):
            x = layer(x, valid_mask, attn_mask)

        statistics = self.statistics_projection(x)
        statistics = statistics.masked_fill(padding_bth, 0.0)

        prior_mean, prior_log_scale = statistics.chunk(2, dim=-1)

        encoded_text = x.transpose(1, 2)
        prior_mean = prior_mean.transpose(1, 2)
        prior_log_scale = prior_log_scale.transpose(1, 2)
        text_mask = valid_mask.unsqueeze(1).to(x.dtype)

        return encoded_text, prior_mean, prior_log_scale, text_mask


class LegacyTextEncoder(nn.Module):
    """
    TTS_0.02 text encoder. Converts text token IDs into:

    1. Encoded text features
    2. Prior mean
    3. Prior log-scale
    4. Text mask
    """

    def __init__(
        self,
        vocab_size,
        hidden_channels=192,
        latent_channels=192,
        feedforward_channels=768,
        num_heads=2,
        num_layers=6,
        dropout=0.1,
        max_text_length=512,
        padding_id=0
    ):
        super().__init__()

        self.hidden_channels = hidden_channels
        self.max_text_length = max_text_length

        # Convert token IDs into vectors
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=hidden_channels,
            padding_idx=padding_id
        )

        # Give the Transformer information about token positions
        self.position_embedding = nn.Embedding(
            num_embeddings=max_text_length,
            embedding_dim=hidden_channels
        )

        # One Transformer block:
        # self-attention + feed-forward network
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=hidden_channels,
            nhead=num_heads,
            dim_feedforward=feedforward_channels,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        # Stack multiple Transformer blocks
        self.transformer = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=num_layers
        )

        # Produce mean and log-scale together
        self.statistics_projection = nn.Linear(
            hidden_channels,
            latent_channels * 2
        )

    def forward(self, token_ids, text_lengths):
        """
        token_ids:
            [batch, text_length]

        text_lengths:
            [batch]
        """
        batch_size, text_length = token_ids.shape

        if text_length > self.max_text_length:
            raise ValueError(
                f"Text length {text_length} exceeds "
                f"max_text_length={self.max_text_length}"
            )

        # --------------------------------------------------
        # 1. Create text mask
        # --------------------------------------------------

        valid_mask = create_sequence_mask(
            text_lengths,
            max_length=text_length
        )  # [batch, text_length]

        # PyTorch Transformer uses True to mean:
        # "ignore this padding position"
        padding_mask = ~valid_mask

        # --------------------------------------------------
        # 2. Create token and position embeddings
        # --------------------------------------------------

        token_features = self.token_embedding(token_ids)

        position_ids = torch.arange(
            text_length,
            device=token_ids.device
        ).unsqueeze(0)

        position_features = self.position_embedding(position_ids)

        # [batch, text_length, hidden_channels]
        encoded_text = token_features + position_features

        # --------------------------------------------------
        # 3. Apply the Transformer
        # --------------------------------------------------

        encoded_text = self.transformer(
            encoded_text,
            src_key_padding_mask=padding_mask
        )

        # Set padding positions to zero
        float_mask = valid_mask.unsqueeze(-1).to(encoded_text.dtype)
        encoded_text = encoded_text * float_mask

        # --------------------------------------------------
        # 4. Predict prior distribution statistics
        # --------------------------------------------------

        statistics = self.statistics_projection(encoded_text)

        prior_mean, prior_log_scale = statistics.chunk(
            chunks=2,
            dim=-1
        )

        # VITS usually expects [batch, channels, time]
        encoded_text = encoded_text.transpose(1, 2)
        prior_mean = prior_mean.transpose(1, 2)
        prior_log_scale = prior_log_scale.transpose(1, 2)

        # [batch, 1, text_length]
        text_mask = valid_mask.unsqueeze(1).to(encoded_text.dtype)

        return (
            encoded_text,
            prior_mean,
            prior_log_scale,
            text_mask
        )

## 7. Posterior encoder

During training, the posterior encoder observes the full target spectrogram and predicts $q(z\mid y)$. The latent sample uses the reparameterization trick:

$$z = \mu_q + \epsilon \exp(\log \sigma_q), \qquad \epsilon \sim \mathcal{N}(0, I).$$

**`posterior_encoder = tts010`** (`PosteriorEncoder`, benlu): a 513→256→192 convolutional stem,
then 8 non-causal WaveNet blocks (kernel 3, dilations 1, 2, 4, 8 twice). Each block uses a
tanh × sigmoid gate and emits residual and skip outputs; the skips are summed and scaled by
$1/\sqrt{8}$. $\log \sigma_q$ is clamped to $[-7, 2]$ to keep $\exp$ finite under fp16. In
speaker mode every block adds a projected speaker vector to its gate input.

**`posterior_encoder = tts002`** (`LegacyPosteriorEncoder`): a 1×1 projection and four plain
dilated convolutions (kernel 5, dilations 1, 2, 4, 8) with GELU; the speaker vector is added
once at the input.

In [ ]:
class GatedResidualBlock(nn.Module):
    """
    WaveNet-style non-causal residual block.

    x: [batch, channels, time]; mask: [batch, 1, time];
    condition: optional [batch, condition_channels, 1] or [batch, condition_channels, time].
    """

    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout=0.1,
        condition_channels=0,
    ):
        super().__init__()

        if kernel_size % 2 == 0:
            raise ValueError(
                "kernel_size must be odd."
            )

        padding = (
            dilation
            * (kernel_size - 1)
            // 2
        )

        # Produce 2 * channels: one half for filter values, one half for gate values.
        self.temporal_conv = nn.Conv1d(
            in_channels=channels,
            out_channels=2 * channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
        )

        # Optional speaker conditioning
        if condition_channels > 0:
            self.condition_projection = nn.Conv1d(
                in_channels=condition_channels,
                out_channels=2 * channels,
                kernel_size=1,
            )
        else:
            self.condition_projection = None

        # Produce residual and skip outputs
        self.residual_skip_projection = nn.Conv1d(
            in_channels=channels,
            out_channels=2 * channels,
            kernel_size=1,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, condition=None):
        gate_input = self.temporal_conv(
            x * mask
        )

        # Add optional speaker condition
        if condition is not None:
            if self.condition_projection is None:
                raise ValueError(
                    "A condition was provided, but "
                    "condition_channels=0."
                )

            gate_input = (
                gate_input
                + self.condition_projection(
                    condition
                )
            )

        # Gated activation
        filter_values, gate_values = gate_input.chunk(2, dim=1)

        hidden = (
            torch.tanh(filter_values)
            * torch.sigmoid(gate_values)
        )

        hidden = self.dropout(hidden)
        hidden = hidden * mask

        # Residual and skip connections
        residual, skip = (
            self.residual_skip_projection(
                hidden
            ).chunk(2, dim=1)
        )
        x = (x + residual) * mask
        skip = skip * mask
        return x, skip


class PosteriorEncoder(nn.Module):
    """
    TTS_0.10 posterior encoder (benlu). Spectrogram [batch, spec_channels, time] ->
    (z, posterior_mean, posterior_log_scale, spectrogram_mask). g is an optional
    [batch, condition_channels, 1] speaker vector.
    """

    def __init__(
        self,
        spec_channels,
        latent_channels=192,
        hidden_channels=192,
        stem_channels=256,
        kernel_size=3,
        dilations=(
            1, 2, 4, 8,
            1, 2, 4, 8,
        ),
        dropout=0.1,
        condition_channels=0,
        minimum_log_scale=-7.0,
        maximum_log_scale=2.0,
    ):
        super().__init__()

        if len(dilations) == 0:
            raise ValueError(
                "At least one dilation is required."
            )

        self.minimum_log_scale = minimum_log_scale
        self.maximum_log_scale = maximum_log_scale

        # Progressive input stem; for a 513-channel spectrogram: 513 -> 256 -> 192
        self.input_projection = nn.Conv1d(
            spec_channels,
            stem_channels,
            kernel_size=1,
        )

        self.input_refinement = nn.Conv1d(
            stem_channels,
            hidden_channels,
            kernel_size=3,
            padding=1,
        )

        # WaveNet-style residual encoder
        self.blocks = nn.ModuleList([
            GatedResidualBlock(
                channels=hidden_channels,
                kernel_size=kernel_size,
                dilation=dilation,
                dropout=dropout,
                condition_channels=condition_channels,
            )
            for dilation in dilations
        ])

        # Mean and log scale from one projection, then split.
        self.statistics_projection = nn.Conv1d(
            hidden_channels,
            2 * latent_channels,
            kernel_size=1,
        )

    def forward(
        self,
        spectrogram,
        spectrogram_lengths,
        g=None,
        noise_scale=1.0,
        sample=True,
    ):
        time_length = spectrogram.size(2)

        valid_mask = create_sequence_mask(
            spectrogram_lengths,
            max_length=time_length,
        )

        spectrogram_mask = (
            valid_mask
            .unsqueeze(1)
            .to(
                device=spectrogram.device,
                dtype=spectrogram.dtype,
            )
        )

        # Progressive feature extraction
        hidden = self.input_projection(spectrogram)
        hidden = F.silu(hidden) * spectrogram_mask

        hidden = self.input_refinement(hidden)
        hidden = F.silu(hidden) * spectrogram_mask

        # Residual blocks and skip aggregation
        skip_sum = torch.zeros_like(hidden)

        for block in self.blocks:
            hidden, skip = block(
                hidden,
                spectrogram_mask,
                g,
            )
            skip_sum = skip_sum + skip

        # Keep activation magnitude reasonably stable
        hidden = (skip_sum / math.sqrt(len(self.blocks)))
        hidden = F.silu(hidden) * spectrogram_mask

        # Posterior distribution
        statistics = (
            self.statistics_projection(hidden)
            * spectrogram_mask
        )
        posterior_mean, posterior_log_scale = statistics.chunk(2, dim=1)

        # Prevent exp(log_scale) from overflowing, particularly during early AMP training.
        posterior_log_scale = (
            posterior_log_scale.clamp(
                min=self.minimum_log_scale,
                max=self.maximum_log_scale,
            )
            * spectrogram_mask
        )

        # Reparameterization
        if sample:
            noise = torch.randn_like(posterior_mean)
            posterior_std = torch.exp(posterior_log_scale)

            z = (
                posterior_mean
                + noise_scale
                * noise
                * posterior_std
            )
        else:
            # Useful for deterministic reconstruction
            z = posterior_mean
        z = z * spectrogram_mask

        return (
            z,
            posterior_mean,
            posterior_log_scale,
            spectrogram_mask,
        )


class LegacyPosteriorEncoder(nn.Module):
    """
    TTS_0.02 posterior encoder. Spectrogram [batch, spec_channels, time] ->
    (z, posterior_mean, posterior_log_scale, spectrogram_mask).
    """

    def __init__(
        self,
        spec_channels,
        latent_channels=192,
        hidden_channels=192,
        dropout=0.1,
        gin_channels=0
    ):
        super().__init__()
        # Convert spectrogram channels into hidden channels
        self.input_projection = nn.Conv1d(spec_channels, hidden_channels, kernel_size=1)

        # Dilated convolution layers; padding keeps the time length unchanged.
        self.conv1 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=1,
            padding=2
        )

        self.conv2 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=2,
            padding=4
        )

        self.conv3 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=4,
            padding=8
        )

        self.conv4 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=8,
            padding=16
        )

        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

        # Separate layers make the two outputs easier to understand
        self.mean_projection = nn.Conv1d(
            hidden_channels,
            latent_channels,
            kernel_size=1
        )

        self.log_scale_projection = nn.Conv1d(
            hidden_channels,
            latent_channels,
            kernel_size=1
        )

        self.condition_projection = (
            nn.Conv1d(gin_channels, hidden_channels, 1) if gin_channels > 0 else None
        )

    def forward(self, spectrogram, spectrogram_lengths, g=None):
        time_length = spectrogram.size(2)

        # [batch, time]
        valid_mask = create_sequence_mask(
            spectrogram_lengths,
            max_length=time_length
        )

        # [batch, 1, time]
        spectrogram_mask = valid_mask.unsqueeze(1).to(
            spectrogram.dtype
        )

        # CNN feature extraction
        hidden = self.input_projection(spectrogram)
        if g is not None:
            if self.condition_projection is None:
                raise ValueError("Posterior encoder was built without conditioning.")
            hidden = hidden + self.condition_projection(g)
        hidden = self.activation(hidden) * spectrogram_mask

        hidden = self.conv1(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        hidden = self.conv2(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        hidden = self.conv3(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        hidden = self.conv4(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        # Posterior distribution
        posterior_mean = (
            self.mean_projection(hidden)
            * spectrogram_mask
        )

        posterior_log_scale = (
            self.log_scale_projection(hidden)
            * spectrogram_mask
        )

        # Reparameterization trick: z = mean + noise * standard deviation
        noise = torch.randn_like(posterior_mean)
        posterior_std = torch.exp(posterior_log_scale)

        z = (
            posterior_mean
            + noise * posterior_std
        ) * spectrogram_mask

        return (
            z,
            posterior_mean,
            posterior_log_scale,
            spectrogram_mask
        )

## 8. Waveform decoder

The decoder behaves like a compact neural vocoder. Four transposed-convolution stages increase time resolution by $8\times8\times2\times2=256$, exactly matching the STFT hop length.

**`decoder = tts010`** (`Decoder`, benlu): HiFi-GAN-style multi-receptive-field fusion. After
every upsampling stage, two residual blocks with kernels 3 and 7 (dilations 1, 3, 5) process
the same features and their outputs are averaged. The output convolution has no bias.

**`decoder = tts002`** (`LegacyDecoder`): one kernel-3 residual block per stage.

In [ ]:
class ResBlock(nn.Module):
    """TTS_0.10 residual block (benlu): dilated conv, then a dilation-1 refinement conv."""

    def __init__(
        self,
        channels,
        kernel_size,
        dilations=(1, 3, 5),
        negative_slope=0.1,
    ):
        super().__init__()

        if kernel_size % 2 == 0:
            raise ValueError(
                "kernel_size must be odd."
            )
        self.negative_slope = negative_slope

        # First convolution in each residual pair uses different dilation values.
        self.dilated_convs = nn.ModuleList()

        # Second convolution uses dilation=1 to refine local details.
        self.refinement_convs = nn.ModuleList()

        for dilation in dilations:
            dilated_padding = (dilation * (kernel_size - 1) // 2)

            self.dilated_convs.append(
                nn.Conv1d(
                    in_channels=channels,
                    out_channels=channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    padding=dilated_padding,
                )
            )

            self.refinement_convs.append(
                nn.Conv1d(
                    in_channels=channels,
                    out_channels=channels,
                    kernel_size=kernel_size,
                    dilation=1,
                    padding=(
                        kernel_size - 1
                    ) // 2,
                )
            )

    def forward(self, x):
        for dilated_conv, refinement_conv in zip(
            self.dilated_convs,
            self.refinement_convs,
        ):
            residual = x
            hidden = F.leaky_relu(x, negative_slope=self.negative_slope)
            hidden = dilated_conv(hidden)

            hidden = F.leaky_relu(hidden, negative_slope=self.negative_slope)
            hidden = refinement_conv(hidden)

            x = residual + hidden
        return x


class LegacyResBlock(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=3,
        dilations=(1, 3, 5)
    ):
        """
        TTS_0.02 residual block containing three dilated convolution stages.

        Each stage performs (repeat three times with dilation 1, 3, 5):
        LeakyReLU -> dilated Conv1d -> LeakyReLU -> normal Conv1d -> add original input
        """

        super().__init__()

        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),

                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    dilation=dilation,
                    padding=dilation * (kernel_size - 1) // 2
                ),

                nn.LeakyReLU(0.1),

                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    padding=(kernel_size - 1) // 2
                )
            )
            for dilation in dilations
        ])

    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)

        return x

In [ ]:
class Decoder(nn.Module):
    """
    TTS_0.10 decoder (benlu). Latent [batch, latent_channels, latent_time] ->
    waveform [batch, 1, latent_time * 256]. g is an optional [batch, condition_channels, 1]
    speaker vector.
    """

    def __init__(
        self,
        latent_channels=192,
        initial_channels=512,
        upsample_rates=(8, 8, 2, 2),
        upsample_kernel_sizes=(16, 16, 4, 4),
        resblock_kernel_sizes=(3, 7),
        resblock_dilations=(1, 3, 5),
        condition_channels=0,
        negative_slope=0.1,
    ):
        super().__init__()
        number_of_stages = len(upsample_rates)

        if (initial_channels % (2 ** number_of_stages) != 0):
            raise ValueError(
                "initial_channels must be divisible "
                "by 2 ** number_of_stages."
            )

        self.negative_slope = negative_slope
        self.upsample_factor = math.prod(upsample_rates)

        # Input projection: [B, 192, T] -> [B, 512, T]
        self.input_projection = nn.Conv1d(
            in_channels=latent_channels,
            out_channels=initial_channels,
            kernel_size=7,
            padding=3,
        )

        # Optional speaker conditioning
        if condition_channels > 0:
            self.condition_projection = nn.Conv1d(
                in_channels=condition_channels,
                out_channels=initial_channels,
                kernel_size=1,
            )
        else:
            self.condition_projection = None

        # Upsampling layers and residual blocks
        self.upsample_layers = nn.ModuleList()
        self.resblock_stages = nn.ModuleList()

        current_channels = initial_channels

        for (upsample_rate, upsample_kernel_size) in zip(upsample_rates, upsample_kernel_sizes):
            next_channels = current_channels // 2

            # This padding makes the output length exactly input_length * upsample_rate.
            padding_difference = upsample_kernel_size - upsample_rate

            if padding_difference % 2 != 0:
                raise ValueError(
                    "upsample_kernel_size - "
                    "upsample_rate must be even."
                )
            upsample_padding = padding_difference // 2

            self.upsample_layers.append(
                nn.ConvTranspose1d(
                    in_channels=current_channels,
                    out_channels=next_channels,
                    kernel_size=(
                        upsample_kernel_size
                    ),
                    stride=upsample_rate,
                    padding=upsample_padding,
                )
            )

            # Multiple receptive fields: kernel sizes 3 and 7 operate on the same features.
            stage_resblocks = nn.ModuleList([
                ResBlock(
                    channels=next_channels,
                    kernel_size=kernel_size,
                    dilations=resblock_dilations,
                    negative_slope=negative_slope,
                )
                for kernel_size
                in resblock_kernel_sizes
            ])

            self.resblock_stages.append(
                stage_resblocks
            )
            current_channels = next_channels

        # Output projection: [B, 32, T * 256] -> [B, 1, T * 256]
        self.output_projection = nn.Conv1d(
            in_channels=current_channels,
            out_channels=1,
            kernel_size=7,
            padding=3,
            bias=False,
        )

    def forward(self, z, g=None):
        x = self.input_projection(z)

        # Add target speaker information.
        if g is not None:
            if self.condition_projection is None:
                raise ValueError(
                    "A speaker condition was provided, "
                    "but condition_channels=0."
                )

            x = x + self.condition_projection(g)

        # Progressive waveform upsampling
        for (upsample_layer, stage_resblocks) in zip(self.upsample_layers, self.resblock_stages):
            x = F.leaky_relu(x, negative_slope=(self.negative_slope))
            x = upsample_layer(x)

            # Multi-receptive-field fusion
            stage_output = None

            for resblock in stage_resblocks:
                block_output = resblock(x)

                if stage_output is None:
                    stage_output = block_output
                else:
                    stage_output = stage_output + block_output

            x = stage_output / len(stage_resblocks)

        # Generate waveform
        x = F.leaky_relu(
            x,
            negative_slope=self.negative_slope,
        )
        x = self.output_projection(x)
        waveform = torch.tanh(x)
        return waveform


class LegacyDecoder(nn.Module):
    """
    TTS_0.02 decoder. Latent [batch, latent_channels, latent_time] ->
    waveform [batch, 1, latent_time * 256].
    """

    def __init__(
        self,
        latent_channels=192,
        initial_channels=512,
        gin_channels=0
    ):
        super().__init__()

        # Input projection: [B, 192, T] -> [B, 512, T]
        self.input_projection = nn.Conv1d(
            latent_channels,
            initial_channels,
            kernel_size=7,
            padding=3
        )

        # Upsampling stage 1: time T -> T × 8, channels 512 -> 256
        self.upsample1 = nn.ConvTranspose1d(
            in_channels=512,
            out_channels=256,
            kernel_size=16,
            stride=8,
            padding=4
        )

        self.resblock1 = LegacyResBlock(
            channels=256,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # Upsampling stage 2: time T × 8 -> T × 64, channels 256 -> 128
        self.upsample2 = nn.ConvTranspose1d(
            in_channels=256,
            out_channels=128,
            kernel_size=16,
            stride=8,
            padding=4
        )

        self.resblock2 = LegacyResBlock(
            channels=128,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # Upsampling stage 3: time T × 64 -> T × 128, channels 128 -> 64
        self.upsample3 = nn.ConvTranspose1d(
            in_channels=128,
            out_channels=64,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock3 = LegacyResBlock(
            channels=64,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # Upsampling stage 4: time T × 128 -> T × 256, channels 64 -> 32
        self.upsample4 = nn.ConvTranspose1d(
            in_channels=64,
            out_channels=32,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock4 = LegacyResBlock(
            channels=32,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # Output waveform: [B, 32, T × 256] -> [B, 1, T × 256]
        self.output_projection = nn.Conv1d(
            in_channels=32,
            out_channels=1,
            kernel_size=7,
            padding=3
        )

        self.condition_projection = (
            nn.Conv1d(gin_channels, initial_channels, 1) if gin_channels > 0 else None
        )

    def forward(self, z, g=None):
        # Input projection
        x = self.input_projection(z)
        if g is not None:
            if self.condition_projection is None:
                raise ValueError("Decoder was built without conditioning.")
            x = x + self.condition_projection(g)

        # Upsampling stage 1
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample1(x)
        x = self.resblock1(x)

        # Upsampling stage 2
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample2(x)
        x = self.resblock2(x)

        # Upsampling stage 3
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample3(x)
        x = self.resblock3(x)

        # Upsampling stage 4
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample4(x)
        x = self.resblock4(x)

        # Generate waveform
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.output_projection(x)

        # Restrict waveform values to [-1, 1]
        waveform = torch.tanh(x)
        return waveform

### Random training segments

The encoders process the **complete** text and spectrogram because alignment needs the whole utterance. Only the latent passed to the decoder is cropped. Decoding 32 frames instead of the complete utterance greatly reduces waveform memory use while keeping the encoder context intact.

Because each frame represents 256 samples, a 32-frame latent segment produces exactly 8,192 waveform samples.

In [ ]:
def slice_segments_at(x, start_indices, segment_size):
    """Slice one segment from every item in a [B, C, T] tensor."""
    segments = []

    for batch_index in range(x.size(0)):
        start = int(start_indices[batch_index].item())
        segment = x[
            batch_index : batch_index + 1,
            :,
            start : start + segment_size,
        ]

        # Padding is only needed when an utterance is shorter than the segment.
        missing = segment_size - segment.size(-1)
        if missing > 0:
            segment = F.pad(segment, (0, missing))

        segments.append(segment)

    return torch.cat(segments, dim=0)


def random_segments(x, lengths, segment_size):
    """Select one valid random start position for every batch item."""
    max_start = (lengths - segment_size).clamp(min=0)
    random_numbers = torch.rand(max_start.shape, device=x.device)
    start_indices = (random_numbers * (max_start + 1).float()).long()

    segments = slice_segments_at(x, start_indices, segment_size)
    return segments, start_indices

## 9. Normalizing flow

The posterior encoder and text encoder initially describe different latent distributions. Reversible coupling layers transform the posterior latent into the text-prior space during training and apply the inverse transformation during inference.

This compact version uses additive coupling. Splitting and flipping the channels allows successive layers to update every channel while keeping the transformation invertible.

In [ ]:
class ChannelLayerNorm(nn.Module):
    """LayerNorm for tensors with shape [batch, channels, time]."""

    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        # [B, C, T] -> [B, T, C]
        x = x.transpose(1, 2)

        x = self.norm(x)

        # [B, T, C] -> [B, C, T]
        return x.transpose(1, 2)

class ResidualConvLayer(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout=0.0
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.norm = ChannelLayerNorm(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        residual = x

        x = self.conv(x * mask)
        x = self.norm(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Residual connection
        return (residual + x) * mask

class CouplingNetwork(nn.Module):
    def __init__(
        self,
        input_channels,
        hidden_channels,
        kernel_size=5,
        dilation_rate=1,
        n_layers=4,
        gin_channels=0
    ):
        super().__init__()

        self.input_projection = nn.Conv1d(
            input_channels,
            hidden_channels,
            kernel_size=1
        )

        self.residual_layers = nn.ModuleList([
            ResidualConvLayer(
                channels=hidden_channels,
                kernel_size=kernel_size,
                dilation=dilation_rate ** layer_index
            )
            for layer_index in range(n_layers)
        ])

        self.output_projection = nn.Conv1d(
            hidden_channels,
            input_channels,
            kernel_size=1
        )

        # Speaker embedding projection
        if gin_channels > 0:
            self.condition_projection = nn.Conv1d(
                gin_channels,
                hidden_channels,
                kernel_size=1
            )
        else:
            self.condition_projection = None

        # Make the coupling layer start as an identity function
        nn.init.zeros_(self.output_projection.weight)
        nn.init.zeros_(self.output_projection.bias)

    def forward(self, x, mask, g=None):
        x = self.input_projection(x * mask)

        if g is not None:
            if self.condition_projection is None:
                raise ValueError(
                    "gin_channels must be greater than 0 when using g."
                )

            # g is normally [B, gin_channels, 1]
            # It broadcasts across the time dimension
            x = x + self.condition_projection(g)

        x = x * mask

        for layer in self.residual_layers:
            x = layer(x, mask)

        shift = self.output_projection(x)

        return shift * mask

class ResidualCouplingLayer(nn.Module):
    def __init__(
        self,
        channels,
        hidden_channels,
        kernel_size=5,
        dilation_rate=1,
        n_layers=4,
        gin_channels=0
    ):
        super().__init__()

        if channels % 2 != 0:
            raise ValueError("channels must be an even number.")

        half_channels = channels // 2

        self.coupling_network = CouplingNetwork(
            input_channels=half_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            n_layers=n_layers,
            gin_channels=gin_channels
        )

    def forward(self, x, mask, g=None, reverse=False):
        # Split channels into two equal parts
        x_keep, x_change = torch.chunk(x, chunks=2, dim=1)

        # Predict the shift using only x_keep
        shift = self.coupling_network(x_keep, mask, g)

        if reverse:
            # Inverse transformation
            x_change = x_change - shift

            x = torch.cat([x_keep, x_change], dim=1)
            return x * mask

        # Forward transformation
        x_change = x_change + shift

        x = torch.cat([x_keep, x_change], dim=1)
        x = x * mask

        # An additive coupling layer has log determinant = 0
        log_determinant = torch.zeros(
            x.size(0),
            device=x.device,
            dtype=x.dtype
        )

        return x, log_determinant

class ResidualCouplingBlock(nn.Module):
    def __init__(
        self,
        channels,
        hidden_channels,
        kernel_size=5,
        dilation_rate=1,
        n_layers=4,
        n_flows=4,
        gin_channels=0
    ):
        super().__init__()

        self.coupling_layers = nn.ModuleList([
            ResidualCouplingLayer(
                channels=channels,
                hidden_channels=hidden_channels,
                kernel_size=kernel_size,
                dilation_rate=dilation_rate,
                n_layers=n_layers,
                gin_channels=gin_channels
            )
            for _ in range(n_flows)
        ])

    def forward(self, x, mask, g=None, reverse=False):
        if not reverse:
            for coupling_layer in self.coupling_layers:
                x, _ = coupling_layer(
                    x,
                    mask,
                    g=g,
                    reverse=False
                )

                # Replace modules.Flip()
                x = torch.flip(x, dims=[1])

        else:
            # Undo operations in the opposite order
            for coupling_layer in reversed(self.coupling_layers):
                x = torch.flip(x, dims=[1])

                x = coupling_layer(
                    x,
                    mask,
                    g=g,
                    reverse=True
                )

        return x

## 10. Duration predictor

The duration predictor estimates the log number of spectrogram frames assigned to each text token. Its input is detached so the duration objective trains the predictor without directly changing the text encoder.

During training, MAS supplies target durations. During inference, the predicted durations determine the output speech length.

In [ ]:
class DurationConvBlock(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        kernel_size,
        dropout
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            input_channels,
            output_channels,
            kernel_size,
            padding=kernel_size // 2
        )

        self.norm = ChannelLayerNorm(output_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.conv(x * mask)
        x = F.relu(x)
        x = self.norm(x)
        x = self.dropout(x)

        return x * mask

class DurationPredictor(nn.Module):
    def __init__(
        self,
        input_channels,
        hidden_channels=256,
        kernel_size=3,
        dropout=0.5,
        gin_channels=0
    ):
        super().__init__()

        self.conv_block_1 = DurationConvBlock(
            input_channels,
            hidden_channels,
            kernel_size,
            dropout
        )

        self.conv_block_2 = DurationConvBlock(
            hidden_channels,
            hidden_channels,
            kernel_size,
            dropout
        )

        # Produce one duration value for each text token
        self.output_projection = nn.Conv1d(
            hidden_channels,
            out_channels=1,
            kernel_size=1
        )

        if gin_channels > 0:
            self.condition_projection = nn.Conv1d(
                gin_channels,
                input_channels,
                kernel_size=1
            )
        else:
            self.condition_projection = None

    def forward(self, x, mask, g=None):
        # Prevent duration loss from updating the text encoder
        x = x.detach()

        if g is not None:
            if self.condition_projection is None:
                raise ValueError(
                    "gin_channels must be greater than 0 when using g."
                )

            g = g.detach()
            x = x + self.condition_projection(g)

        x = self.conv_block_1(x, mask)
        x = self.conv_block_2(x, mask)

        log_duration = self.output_projection(x)

        return log_duration * mask


# ---- TTS_0.20 option: the stochastic duration predictor of VITS (Kim et al., 2021). ----
# Rational-quadratic spline (Durkan et al., 2019), as used by the VITS reference implementation.
# Minimum bin width, bin height and derivative: 1e-3 each, as in the reference implementation.


def spline_searchsorted(bin_locations, inputs, eps=1e-6):
    bin_locations = bin_locations.clone()
    bin_locations[..., -1] += eps
    return torch.sum(inputs[..., None] >= bin_locations, dim=-1) - 1


def rational_quadratic_spline(inputs, unnormalized_widths, unnormalized_heights, unnormalized_derivatives,
                              inverse=False, left=0.0, right=1.0, bottom=0.0, top=1.0):
    num_bins = unnormalized_widths.shape[-1]
    widths = F.softmax(unnormalized_widths, dim=-1)
    widths = 1e-3 + (1 - 1e-3 * num_bins) * widths
    cumwidths = F.pad(torch.cumsum(widths, dim=-1), pad=(1, 0), mode="constant", value=0.0)
    cumwidths = (right - left) * cumwidths + left
    cumwidths[..., 0] = left
    cumwidths[..., -1] = right
    widths = cumwidths[..., 1:] - cumwidths[..., :-1]

    derivatives = 1e-3 + F.softplus(unnormalized_derivatives)

    heights = F.softmax(unnormalized_heights, dim=-1)
    heights = 1e-3 + (1 - 1e-3 * num_bins) * heights
    cumheights = F.pad(torch.cumsum(heights, dim=-1), pad=(1, 0), mode="constant", value=0.0)
    cumheights = (top - bottom) * cumheights + bottom
    cumheights[..., 0] = bottom
    cumheights[..., -1] = top
    heights = cumheights[..., 1:] - cumheights[..., :-1]

    bin_idx = spline_searchsorted(cumheights if inverse else cumwidths, inputs)[..., None]
    input_cumwidths = cumwidths.gather(-1, bin_idx)[..., 0]
    input_bin_widths = widths.gather(-1, bin_idx)[..., 0]
    input_cumheights = cumheights.gather(-1, bin_idx)[..., 0]
    delta = heights / widths
    input_delta = delta.gather(-1, bin_idx)[..., 0]
    input_derivatives = derivatives.gather(-1, bin_idx)[..., 0]
    input_derivatives_plus_one = derivatives[..., 1:].gather(-1, bin_idx)[..., 0]
    input_heights = heights.gather(-1, bin_idx)[..., 0]
    slope_sum = input_derivatives + input_derivatives_plus_one - 2 * input_delta

    if inverse:
        a = (inputs - input_cumheights) * slope_sum + input_heights * (input_delta - input_derivatives)
        b = input_heights * input_derivatives - (inputs - input_cumheights) * slope_sum
        c = -input_delta * (inputs - input_cumheights)
        discriminant = (b.pow(2) - 4 * a * c).clamp(min=0.0)
        root = (2 * c) / (-b - torch.sqrt(discriminant))
        outputs = root * input_bin_widths + input_cumwidths
        theta_one_minus_theta = root * (1 - root)
        denominator = input_delta + slope_sum * theta_one_minus_theta
        derivative_numerator = input_delta.pow(2) * (input_derivatives_plus_one * root.pow(2)
                                                     + 2 * input_delta * theta_one_minus_theta
                                                     + input_derivatives * (1 - root).pow(2))
        return outputs, -(torch.log(derivative_numerator) - 2 * torch.log(denominator))

    theta = (inputs - input_cumwidths) / input_bin_widths
    theta_one_minus_theta = theta * (1 - theta)
    numerator = input_heights * (input_delta * theta.pow(2) + input_derivatives * theta_one_minus_theta)
    denominator = input_delta + slope_sum * theta_one_minus_theta
    outputs = input_cumheights + numerator / denominator
    derivative_numerator = input_delta.pow(2) * (input_derivatives_plus_one * theta.pow(2)
                                                 + 2 * input_delta * theta_one_minus_theta
                                                 + input_derivatives * (1 - theta).pow(2))
    return outputs, torch.log(derivative_numerator) - 2 * torch.log(denominator)


def piecewise_rational_quadratic_transform(inputs, unnormalized_widths, unnormalized_heights,
                                           unnormalized_derivatives, inverse=False, tail_bound=5.0):
    """Spline on [-tail_bound, tail_bound], identity (linear tails) outside."""
    inside = (inputs >= -tail_bound) & (inputs <= tail_bound)
    outputs = inputs.clone()
    logabsdet = torch.zeros_like(inputs)
    unnormalized_derivatives = F.pad(unnormalized_derivatives, pad=(1, 1))
    constant = math.log(math.exp(1 - 1e-3) - 1)
    unnormalized_derivatives[..., 0] = constant
    unnormalized_derivatives[..., -1] = constant
    if inside.any():
        outputs[inside], logabsdet[inside] = rational_quadratic_spline(
            inputs[inside], unnormalized_widths[inside, :], unnormalized_heights[inside, :],
            unnormalized_derivatives[inside, :], inverse=inverse,
            left=-tail_bound, right=tail_bound, bottom=-tail_bound, top=tail_bound)
    return outputs, logabsdet


class DilatedDepthSeparableConv(nn.Module):
    def __init__(self, channels, kernel_size, n_layers, dropout=0.0):
        super().__init__()
        self.convs_sep = nn.ModuleList()
        self.convs_1x1 = nn.ModuleList()
        self.norms_1 = nn.ModuleList()
        self.norms_2 = nn.ModuleList()
        for i in range(n_layers):
            dilation = kernel_size ** i
            padding = (kernel_size * dilation - dilation) // 2
            self.convs_sep.append(nn.Conv1d(channels, channels, kernel_size, groups=channels,
                                            dilation=dilation, padding=padding))
            self.convs_1x1.append(nn.Conv1d(channels, channels, 1))
            self.norms_1.append(ChannelLayerNorm(channels))
            self.norms_2.append(ChannelLayerNorm(channels))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, x_mask, g=None):
        if g is not None:
            x = x + g
        for conv_sep, conv_1x1, norm_1, norm_2 in zip(self.convs_sep, self.convs_1x1, self.norms_1, self.norms_2):
            y = F.gelu(norm_1(conv_sep(x * x_mask)))
            y = self.dropout(F.gelu(norm_2(conv_1x1(y))))
            x = x + y
        return x * x_mask


class LogFlow(nn.Module):
    def forward(self, x, x_mask, reverse=False, **kwargs):
        if not reverse:
            y = torch.log(torch.clamp_min(x, 1e-5)) * x_mask
            return y, torch.sum(-y, [1, 2])
        return torch.exp(x) * x_mask


class FlipFlow(nn.Module):
    def forward(self, x, *args, reverse=False, **kwargs):
        x = torch.flip(x, [1])
        if not reverse:
            return x, torch.zeros(x.size(0), dtype=x.dtype, device=x.device)
        return x


class ElementwiseAffineFlow(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.m = nn.Parameter(torch.zeros(channels, 1))
        self.logs = nn.Parameter(torch.zeros(channels, 1))

    def forward(self, x, x_mask, reverse=False, **kwargs):
        if not reverse:
            y = (self.m + torch.exp(self.logs) * x) * x_mask
            return y, torch.sum(self.logs * x_mask, [1, 2])
        return (x - self.m) * torch.exp(-self.logs) * x_mask


class ConvSplineFlow(nn.Module):
    def __init__(self, in_channels, filter_channels, kernel_size, n_layers, num_bins=10, tail_bound=5.0):
        super().__init__()
        self.filter_channels = filter_channels
        self.num_bins = num_bins
        self.tail_bound = tail_bound
        self.half_channels = in_channels // 2
        self.pre = nn.Conv1d(self.half_channels, filter_channels, 1)
        self.convs = DilatedDepthSeparableConv(filter_channels, kernel_size, n_layers)
        self.proj = nn.Conv1d(filter_channels, self.half_channels * (num_bins * 3 - 1), 1)
        self.proj.weight.data.zero_()
        self.proj.bias.data.zero_()

    def forward(self, x, x_mask, g=None, reverse=False):
        x0, x1 = torch.split(x, [self.half_channels] * 2, 1)
        h = self.proj(self.convs(self.pre(x0), x_mask, g=g)) * x_mask
        b, c, t = x0.shape
        h = h.reshape(b, c, -1, t).permute(0, 1, 3, 2)
        scale = math.sqrt(self.filter_channels)
        x1, logabsdet = piecewise_rational_quadratic_transform(
            x1, h[..., :self.num_bins] / scale, h[..., self.num_bins:2 * self.num_bins] / scale,
            h[..., 2 * self.num_bins:], inverse=reverse, tail_bound=self.tail_bound)
        x = torch.cat([x0, x1], 1) * x_mask
        if not reverse:
            return x, torch.sum(logabsdet * x_mask, [1, 2])
        return x


class StochasticDurationPredictor(nn.Module):
    """
    Flow-based duration model of VITS: forward() returns the per-utterance negative log-likelihood
    (variational bound) of the MAS durations; reverse=True samples log durations. Runs in float32.
    """

    def __init__(self, input_channels, filter_channels=192, kernel_size=3, dropout=0.5, n_flows=4,
                 gin_channels=0):
        super().__init__()
        filter_channels = input_channels
        self.log_flow = LogFlow()
        self.flows = nn.ModuleList([ElementwiseAffineFlow(2)])
        for _ in range(n_flows):
            self.flows.append(ConvSplineFlow(2, filter_channels, kernel_size, n_layers=3))
            self.flows.append(FlipFlow())
        self.post_pre = nn.Conv1d(1, filter_channels, 1)
        self.post_proj = nn.Conv1d(filter_channels, filter_channels, 1)
        self.post_convs = DilatedDepthSeparableConv(filter_channels, kernel_size, n_layers=3, dropout=dropout)
        self.post_flows = nn.ModuleList([ElementwiseAffineFlow(2)])
        for _ in range(4):
            self.post_flows.append(ConvSplineFlow(2, filter_channels, kernel_size, n_layers=3))
            self.post_flows.append(FlipFlow())
        self.pre = nn.Conv1d(input_channels, filter_channels, 1)
        self.proj = nn.Conv1d(filter_channels, filter_channels, 1)
        self.convs = DilatedDepthSeparableConv(filter_channels, kernel_size, n_layers=3, dropout=dropout)
        self.cond = nn.Conv1d(gin_channels, filter_channels, 1) if gin_channels > 0 else None

    def forward(self, x, x_mask, w=None, g=None, reverse=False, noise_scale=1.0):
        with torch.autocast(x.device.type, enabled=False):
            x, x_mask = x.detach().float(), x_mask.float()
            x = self.pre(x)
            if g is not None:
                if self.cond is None:
                    raise ValueError("gin_channels must be greater than 0 when using g.")
                x = x + self.cond(g.detach().float())
            x = self.proj(self.convs(x, x_mask)) * x_mask

            if reverse:
                flows = list(reversed(self.flows))
                flows = flows[:-2] + [flows[-1]]  # drop the unused flow, as in VITS
                z = torch.randn(x.size(0), 2, x.size(2), device=x.device, dtype=x.dtype) * noise_scale
                for flow in flows:
                    z = flow(z, x_mask, g=x, reverse=True)
                return z[:, :1] * x_mask  # log durations, [B, 1, T]

            w = w.float()
            h_w = self.post_proj(self.post_convs(self.post_pre(w), x_mask)) * x_mask
            e_q = torch.randn(w.size(0), 2, w.size(2), device=x.device, dtype=x.dtype) * x_mask
            z_q, logdet_tot_q = e_q, 0.0
            for flow in self.post_flows:
                z_q, logdet_q = flow(z_q, x_mask, g=x + h_w)
                logdet_tot_q = logdet_tot_q + logdet_q
            z_u, z1 = torch.split(z_q, [1, 1], 1)
            u = torch.sigmoid(z_u) * x_mask
            z0 = (w - u) * x_mask
            logdet_tot_q = logdet_tot_q + torch.sum((F.logsigmoid(z_u) + F.logsigmoid(-z_u)) * x_mask, [1, 2])
            logq = torch.sum(-0.5 * (math.log(2 * math.pi) + e_q.pow(2)) * x_mask, [1, 2]) - logdet_tot_q

            z0, logdet_tot = self.log_flow(z0, x_mask)
            z = torch.cat([z0, z1], 1)
            for flow in self.flows:
                z, logdet = flow(z, x_mask, g=x)
                logdet_tot = logdet_tot + logdet
            nll = torch.sum(0.5 * (math.log(2 * math.pi) + z.pow(2)) * x_mask, [1, 2]) - logdet_tot
            return nll + logq  # [B]

## 11. Monotonic Alignment Search (MAS)

Text tokens and spectrogram frames have different lengths, so the model needs a monotonic path connecting them. MAS finds the highest-scoring path subject to two constraints: the alignment moves forward in time and every audio frame is assigned to one text token.

The alignment is a discrete search, so it is computed under `torch.no_grad()`. TTS_0.10 runs
the same dynamic programme vectorised over the whole batch and the text axis, looping only
over frames. It returns paths identical to the original double loop (checked against
benlu's `monotonic_align.py`, including ties) and was about 6× faster than that loop on a
batch of 16 LJSpeech-length utterances; one training step took 0.24 s instead of 0.38 s on an
RTX 4070 Ti SUPER.

In [ ]:
def maximum_path(neg_cent, mask):
    """
    Batch-vectorised monotonic alignment search.

    neg_cent: [b, t_t, t_s] alignment scores (frames x tokens)
    mask:     [b, t_t, t_s] valid positions
    Returns the hard path as a tensor with neg_cent's device and dtype. The dynamic
    programme and tie-breaking are those of the original per-item double loop.
    """
    device, dtype = neg_cent.device, neg_cent.dtype
    neg = np.float32(-1e9)
    value = neg_cent.detach().cpu().numpy().astype(np.float32)
    t_ys = mask.sum(1)[:, 0].detach().cpu().numpy().astype(np.int64)
    t_xs = mask.sum(2)[:, 0].detach().cpu().numpy().astype(np.int64)
    batch, max_frames, max_tokens = value.shape
    xs = np.arange(max_tokens)
    first_row = np.where(xs == 0, np.float32(0.0), neg).astype(np.float32)

    # Forward pass: accumulate the best score inside each item's valid band.
    for y in range(int(t_ys.max())):
        low = np.maximum(0, t_xs + y - t_ys)
        high = np.minimum(t_xs, y + 1)
        band = (xs[None] >= low[:, None]) & (xs[None] < high[:, None]) & (y < t_ys)[:, None]
        if y == 0:
            best = np.broadcast_to(first_row, (batch, max_tokens))
        else:
            previous = value[:, y - 1, :]
            stay = np.where(xs[None] == y, neg, previous)
            advance = np.concatenate([np.full((batch, 1), neg, np.float32), previous[:, :-1]], axis=1)
            best = np.maximum(advance, stay)
        value[:, y, :] = np.where(band, value[:, y, :] + best, value[:, y, :])

    # Backtrack from the last token of the last frame.
    path = np.zeros((batch, max_frames, max_tokens), dtype=np.int32)
    index = t_xs - 1
    rows = np.arange(batch)
    for y in range(max_frames - 1, -1, -1):
        active = y < t_ys
        path[rows[active], y, index[active]] = 1
        if y == 0:
            break
        current = value[rows, y - 1, index]
        left = value[rows, y - 1, np.maximum(index - 1, 0)]
        move = active & (index != 0) & ((index == y) | (current < left))
        index = index - move
    return torch.from_numpy(path).to(device=device, dtype=dtype)

In [ ]:
def expand_with_alignment(values, attention):
    """
    Expand [B, C, T_text] values into [B, C, T_spec].

    attention has shape [B, 1, T_spec, T_text].
    """
    return torch.bmm(
        attention.squeeze(1),
        values.transpose(1, 2),
    ).transpose(1, 2)


def make_inference_path(durations, attention_mask):
    """Convert integer token durations into a hard attention path."""
    end_frames = torch.cumsum(durations, dim=-1)
    start_frames = F.pad(end_frames[..., :-1], (1, 0))

    output_frames = attention_mask.size(2)
    frame_ids = torch.arange(
        output_frames,
        device=durations.device,
    ).view(1, 1, output_frames, 1)

    path = (
        (frame_ids >= start_frames.unsqueeze(2))
        & (frame_ids < end_frames.unsqueeze(2))
    )

    return path.to(attention_mask.dtype) * attention_mask


def gaussian_alignment_score(z, prior_mean, prior_log_scale):
    """
    Compute log p(z_frame | text_token) for every frame/token pair.

    Returns [B, T_spec, T_text].
    """
    inverse_variance = torch.exp(-2.0 * prior_log_scale)

    constant = torch.sum(
        -0.5 * math.log(2.0 * math.pi) - prior_log_scale,
        dim=1,
        keepdim=True,
    )

    z_squared = torch.matmul(
        -0.5 * z.pow(2).transpose(1, 2),
        inverse_variance,
    )

    cross_term = torch.matmul(
        z.transpose(1, 2),
        prior_mean * inverse_variance,
    )

    mean_squared = torch.sum(
        -0.5 * prior_mean.pow(2) * inverse_variance,
        dim=1,
        keepdim=True,
    )

    return constant + z_squared + cross_term + mean_squared

## 12. Complete VITS-style model

The training forward pass encodes text and target audio, maps the posterior latent through the flow, finds the MAS alignment, predicts token durations, and decodes one random latent segment.

Inference uses text only: predicted durations create an alignment, a latent sample is drawn from the expanded prior, the flow is inverted, and the decoder generates the waveform.

`VITSModel(text_encoder=..., posterior_encoder=..., decoder=...)` builds the TTS_0.10
(`"tts010"`) or TTS_0.02 (`"tts002"`) implementation of each component. Attribute names
(`text_encoder`, `posterior_encoder`, `flow`, `duration_predictor`, `decoder`) and the
output dictionary are the same for every choice.

In [ ]:
class VITSModel(nn.Module):
    """VITS with switchable TTS_0.10/TTS_0.02 components and optional speaker-ID conditioning."""

    COMPONENT_CHOICES = ("tts010", "tts002")

    def __init__(
        self,
        vocab_size,
        spec_channels,
        padding_id=0,
        latent_channels=192,
        hidden_channels=192,
        feedforward_channels=768,
        decoder_channels=512,
        segment_samples=8192,
        hop_length=256,
        n_speakers=0,
        gin_channels=0,
        text_encoder="tts010",
        posterior_encoder="tts010",
        decoder="tts010",
        duration_predictor="deterministic",
        encoder_window=None,
        space_id=None,
    ):
        super().__init__()

        if segment_samples % hop_length != 0:
            raise ValueError("segment_samples must be divisible by hop_length.")
        self.components = {"text_encoder": text_encoder, "posterior_encoder": posterior_encoder,
                           "decoder": decoder}
        if any(choice not in self.COMPONENT_CHOICES for choice in self.components.values()):
            raise ValueError(f"Component choices must be one of {self.COMPONENT_CHOICES}: {self.components}")

        if (isinstance(n_speakers, bool) or not isinstance(n_speakers, int) or n_speakers < 0
                or isinstance(gin_channels, bool) or not isinstance(gin_channels, int) or gin_channels < 0
                or bool(n_speakers) != bool(gin_channels)):
            raise ValueError("Use n_speakers=gin_channels=0, or positive integer values for both.")
        self.n_speakers = n_speakers
        self.gin_channels = gin_channels
        self.speaker_embedding = nn.Embedding(n_speakers, gin_channels) if n_speakers else None
        self.hop_length = hop_length
        self.segment_samples = segment_samples
        self.segment_frames = segment_samples // hop_length

        text_encoder_class = TextEncoder if text_encoder == "tts010" else LegacyTextEncoder
        window_arguments = {}
        if encoder_window is not None:
            if text_encoder != "tts010":
                raise ValueError("The windowed encoder option needs the tts010 text encoder.")
            window_arguments = {"window": encoder_window, "space_id": space_id}
        self.text_encoder = text_encoder_class(
            vocab_size=vocab_size,
            hidden_channels=hidden_channels,
            latent_channels=latent_channels,
            feedforward_channels=feedforward_channels,
            padding_id=padding_id,
            **window_arguments,
        )

        if posterior_encoder == "tts010":
            self.posterior_encoder = PosteriorEncoder(
                spec_channels=spec_channels,
                latent_channels=latent_channels,
                hidden_channels=hidden_channels,
                condition_channels=gin_channels,
            )
        else:
            self.posterior_encoder = LegacyPosteriorEncoder(
                spec_channels=spec_channels,
                latent_channels=latent_channels,
                hidden_channels=hidden_channels,
                gin_channels=gin_channels,
            )

        self.flow = ResidualCouplingBlock(
            channels=latent_channels,
            hidden_channels=hidden_channels,
            gin_channels=gin_channels,
        )

        if duration_predictor not in ("deterministic", "stochastic"):
            raise ValueError(f"Unknown duration predictor {duration_predictor!r}.")
        self.stochastic_duration = duration_predictor == "stochastic"
        if self.stochastic_duration:
            self.duration_predictor = StochasticDurationPredictor(
                input_channels=hidden_channels,
                gin_channels=gin_channels,
            )
        else:
            self.duration_predictor = DurationPredictor(
                input_channels=hidden_channels,
                gin_channels=gin_channels,
            )

        if decoder == "tts010":
            self.decoder = Decoder(
                latent_channels=latent_channels,
                initial_channels=decoder_channels,
                condition_channels=gin_channels,
            )
        else:
            self.decoder = LegacyDecoder(
                latent_channels=latent_channels,
                initial_channels=decoder_channels,
                gin_channels=gin_channels,
            )

    def speaker_condition(self, speaker_ids, batch_size):
        """Validate once before embedding lookup; never guess a speaker at inference."""
        if self.speaker_embedding is None:
            if speaker_ids is not None:
                raise ValueError("Speaker IDs supplied to an unconditioned model.")
            return None
        if speaker_ids is None:
            raise ValueError("speaker_ids are required for a conditioned model.")
        if (not isinstance(speaker_ids, torch.Tensor) or speaker_ids.dtype != torch.long
                or speaker_ids.shape != (batch_size,)):
            raise ValueError("speaker_ids must be a torch.long tensor of shape [batch].")
        if speaker_ids.device != self.speaker_embedding.weight.device:
            raise ValueError("speaker_ids and model must be on the same device.")
        if ((speaker_ids < 0) | (speaker_ids >= self.n_speakers)).any():
            raise ValueError("speaker_ids are outside the configured vocabulary.")
        return self.speaker_embedding(speaker_ids).unsqueeze(-1)

    def forward(
        self,
        text_tokens,
        text_lengths,
        spectrogram,
        spectrogram_lengths,
        speaker_ids=None,
    ):
        g = self.speaker_condition(speaker_ids, text_tokens.size(0))
        (
            text_features,
            prior_mean,
            prior_log_scale,
            text_mask,
        ) = self.text_encoder(text_tokens, text_lengths)

        (
            z,
            posterior_mean,
            posterior_log_scale,
            spectrogram_mask,
        ) = self.posterior_encoder(spectrogram, spectrogram_lengths, g=g)

        # Posterior latent space -> text-prior latent space
        z_prior = self.flow(z, spectrogram_mask, g=g)

        # MAS is a hard, non-differentiable alignment search.
        with torch.no_grad():
            score = gaussian_alignment_score(
                z_prior.float(),
                prior_mean.float(),
                prior_log_scale.float(),
            )

            alignment_mask = (
                spectrogram_mask.float().transpose(1, 2)
                * text_mask.float()
            )

            attention = maximum_path(
                score,
                alignment_mask,
            ).unsqueeze(1)

            attention = attention.to(text_features.dtype)

        # Number of spectrogram frames assigned to each text token
        target_duration = attention.sum(dim=2)
        target_log_duration = (
            torch.log(target_duration + 1e-6) * text_mask
        )

        duration_nll = None
        if self.stochastic_duration:
            nll = self.duration_predictor(text_features, text_mask, w=target_duration, g=g)
            duration_nll = nll.sum() / text_mask.float().sum().clamp(min=1.0)
            predicted_log_duration = target_log_duration.detach()  # for logging shapes only
        else:
            predicted_log_duration = self.duration_predictor(
                text_features,
                text_mask,
                g=g,
            )

        # Expand the text prior to spectrogram time.
        expanded_prior_mean = expand_with_alignment(
            prior_mean,
            attention,
        )

        expanded_prior_log_scale = expand_with_alignment(
            prior_log_scale,
            attention,
        )

        # Decode only a short random segment to reduce memory usage.
        z_segment, start_frames = random_segments(
            z,
            spectrogram_lengths,
            self.segment_frames,
        )

        generated_audio = self.decoder(z_segment, g=g)

        return {
            "generated_audio": generated_audio,
            "start_frames": start_frames,
            "attention": attention,
            "text_mask": text_mask,
            "spectrogram_mask": spectrogram_mask,
            "predicted_log_duration": predicted_log_duration,
            "target_log_duration": target_log_duration,
            "z_prior": z_prior,
            "posterior_mean": posterior_mean,
            "posterior_log_scale": posterior_log_scale,
            "prior_mean": expanded_prior_mean,
            "prior_log_scale": expanded_prior_log_scale,
            **({"duration_nll": duration_nll} if duration_nll is not None else {}),
        }

    def predict_log_duration(self, text_features, text_mask, g=None, noise_scale_w=0.8):
        """Log durations at inference (the stochastic predictor samples with noise_scale_w)."""
        if self.stochastic_duration:
            return self.duration_predictor(text_features, text_mask, g=g, reverse=True,
                                           noise_scale=noise_scale_w)
        return self.duration_predictor(text_features, text_mask, g=g)

    @torch.inference_mode()
    def infer(
        self,
        text_tokens,
        text_lengths,
        noise_scale=1.0,
        length_scale=1.0,
        max_frames=None,
        max_duration_per_token=50,
        speaker_ids=None,
        noise_scale_w=0.8,
    ):
        g = self.speaker_condition(speaker_ids, text_tokens.size(0))
        (
            text_features,
            prior_mean,
            prior_log_scale,
            text_mask,
        ) = self.text_encoder(text_tokens, text_lengths)

        predicted_log_duration = self.predict_log_duration(
            text_features,
            text_mask,
            g=g,
            noise_scale_w=noise_scale_w,
        )

        # The clamp prevents an early, untrained model from producing
        # an extremely long waveform and exhausting GPU memory.
        maximum_log_duration = math.log(max_duration_per_token)
        predicted_log_duration = predicted_log_duration.clamp(
            min=-5.0,
            max=maximum_log_duration,
        )

        durations = torch.ceil(
            torch.exp(predicted_log_duration) * length_scale
        ) * text_mask

        output_lengths = durations.sum(dim=(1, 2)).long().clamp(min=1)

        if max_frames is not None:
            output_lengths = output_lengths.clamp(max=max_frames)

        output_frames = int(output_lengths.max().item())

        spectrogram_mask = create_sequence_mask(
            output_lengths,
            output_frames,
        ).unsqueeze(1).to(text_features.dtype)

        attention_mask = (
            spectrogram_mask.unsqueeze(-1)
            * text_mask.unsqueeze(2)
        )

        attention = make_inference_path(
            durations,
            attention_mask,
        )

        expanded_prior_mean = expand_with_alignment(
            prior_mean,
            attention,
        )

        expanded_prior_log_scale = expand_with_alignment(
            prior_log_scale,
            attention,
        )

        noise = torch.randn_like(expanded_prior_mean)
        z_prior = (
            expanded_prior_mean
            + noise
            * torch.exp(expanded_prior_log_scale)
            * noise_scale
        )

        z = self.flow(
            z_prior,
            spectrogram_mask,
            reverse=True,
            g=g,
        )

        generated_audio = self.decoder(z * spectrogram_mask, g=g)
        audio_lengths = output_lengths * self.hop_length

        return {
            "generated_audio": generated_audio,
            "audio_lengths": audio_lengths,
            "attention": attention,
        }

## 13. Training objectives and discriminators

Every arm uses the duration loss (weight 1), the KL loss (weight 1) and a least-squares GAN
generator loss (weight 1), summed over the sub-discriminators.

**`recon_loss = tts010`** (benlu, standard VITS): L1 between Slaney log-mel spectrograms,
weight 45. The target mel comes from the full-utterance training spectrogram, sliced at the
decoded segment; the generated mel uses the same framing.

**`recon_loss = tts002`**: HTK log-mel L1 (weight 1) + multi-resolution STFT loss (weight 0.5)
+ waveform L1 (weight 10).

**`feature_matching = True`**: L1 between discriminator feature maps of real and generated
audio, weight 2 (HiFi-GAN). Real feature maps are detached.

**`discriminator = tts010`** (`MultiPeriodDiscriminator`, benlu/HiFi-GAN): one scale
discriminator plus period discriminators for periods 2, 3, 5, 7, 11, all with weight norm.
**`discriminator = tts002`** (`LegacyDiscriminator`): the TTS_0.02 single 1-D convolution
stack; its LeakyReLU outputs serve as feature maps when feature matching is on.

Spectral and adversarial losses are computed in float32 even when the forward pass uses fp16.

In [ ]:
class MultiResolutionSTFTLoss(nn.Module):
    """Compare generated and real audio at several STFT resolutions (TTS_0.02 loss)."""

    def __init__(
        self,
        resolutions=(
            (512, 128, 512),
            (1024, 256, 1024),
            (2048, 512, 2048),
        ),
        epsilon=1e-7,
    ):
        super().__init__()
        self.resolutions = resolutions
        self.epsilon = epsilon

        for index, (_, _, window_length) in enumerate(resolutions):
            self.register_buffer(
                f"window_{index}",
                torch.hann_window(window_length),
                persistent=False,
            )

    def _magnitude(self, waveform, resolution_index):
        fft_size, hop_length, window_length = self.resolutions[
            resolution_index
        ]

        window = getattr(self, f"window_{resolution_index}")

        stft = torch.stft(
            waveform,
            n_fft=fft_size,
            hop_length=hop_length,
            win_length=window_length,
            window=window,
            return_complex=True,
        )

        return stft.abs().clamp(min=self.epsilon)

    def forward(self, generated, target):
        spectral_convergence = generated.new_tensor(0.0)
        log_magnitude_loss = generated.new_tensor(0.0)

        for index in range(len(self.resolutions)):
            generated_magnitude = self._magnitude(generated, index)
            target_magnitude = self._magnitude(target, index)

            difference_norm = torch.linalg.vector_norm(
                target_magnitude - generated_magnitude,
                dim=(-2, -1),
            )

            target_norm = torch.linalg.vector_norm(
                target_magnitude,
                dim=(-2, -1),
            ).clamp(min=self.epsilon)

            spectral_convergence += (
                difference_norm / target_norm
            ).mean()

            log_magnitude_loss += F.l1_loss(
                torch.log(generated_magnitude),
                torch.log(target_magnitude),
            )

        number_of_resolutions = len(self.resolutions)

        return (
            spectral_convergence / number_of_resolutions,
            log_magnitude_loss / number_of_resolutions,
        )


def discriminator_lsgan_loss(real_outputs, fake_outputs):
    """Least-squares discriminator loss summed over sub-discriminators (float32)."""
    real_loss = sum(((1.0 - real.float()) ** 2).mean() for real in real_outputs)
    fake_loss = sum((fake.float() ** 2).mean() for fake in fake_outputs)
    return {"total": real_loss + fake_loss, "real": real_loss, "fake": fake_loss}


def generator_lsgan_loss(fake_outputs):
    """Least-squares generator loss summed over sub-discriminators (float32)."""
    return sum(((1.0 - fake.float()) ** 2).mean() for fake in fake_outputs)


def feature_matching_loss(real_feature_maps, fake_feature_maps):
    """Unweighted HiFi-GAN feature matching: sum of mean |real - fake| over all layers."""
    loss = 0.0
    for real_maps, fake_maps in zip(real_feature_maps, fake_feature_maps):
        for real, fake in zip(real_maps, fake_maps):
            loss = loss + torch.mean(torch.abs(real.float().detach() - fake.float()))
    return loss


class VITSLoss(nn.Module):
    """Generator objective. recipe "tts010": Slaney log-mel x45; "tts002": mel + STFT + wave."""

    RECIPES = {
        "tts010": {"mel_weight": 45.0, "stft_weight": 0.0, "wave_weight": 0.0},
        "tts002": {"mel_weight": 1.0, "stft_weight": 0.5, "wave_weight": 10.0},
    }

    def __init__(
        self,
        recipe="tts010",
        feature_matching=True,
        sample_rate=22050,
        n_fft=1024,
        hop_length=256,
        win_length=1024,
        n_mels=80,
        duration_weight=1.0,
        kl_weight=1.0,
        gan_weight=1.0,
        feature_matching_weight=2.0,
    ):
        super().__init__()
        if recipe not in self.RECIPES:
            raise ValueError(f"Unknown reconstruction recipe {recipe!r}.")
        self.recipe = recipe
        self.feature_matching = bool(feature_matching)
        self.mel_weight = self.RECIPES[recipe]["mel_weight"]
        self.stft_weight = self.RECIPES[recipe]["stft_weight"]
        self.wave_weight = self.RECIPES[recipe]["wave_weight"]
        self.duration_weight = duration_weight
        self.kl_weight = kl_weight
        self.gan_weight = gan_weight
        self.feature_matching_weight = feature_matching_weight if self.feature_matching else 0.0
        self.audio = {"sample_rate": sample_rate, "n_fft": n_fft, "hop_length": hop_length,
                      "win_length": win_length, "n_mels": n_mels}
        # TTS_0.02 mel: torchaudio defaults (HTK scale, centred frames).
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mels=n_mels,
            power=1.0,
        )
        self.stft_loss = MultiResolutionSTFTLoss()

    def weights(self):
        return {name: getattr(self, name) for name in (
            "mel_weight", "stft_weight", "wave_weight", "duration_weight", "kl_weight",
            "gan_weight", "feature_matching_weight")}

    @staticmethod
    def _duration_loss(outputs):
        if "duration_nll" in outputs:  # TTS_0.20 stochastic duration predictor
            return outputs["duration_nll"].float()
        difference = (
            outputs["predicted_log_duration"].float()
            - outputs["target_log_duration"].float()
        ).pow(2)
        mask = outputs["text_mask"].float()
        return (difference * mask).sum() / mask.sum().clamp(min=1.0)

    @staticmethod
    def _kl_loss(outputs):
        z_prior = outputs["z_prior"].float()
        posterior_log_scale = outputs["posterior_log_scale"].float()
        prior_mean = outputs["prior_mean"].float()
        prior_log_scale = outputs["prior_log_scale"].float()
        mask = outputs["spectrogram_mask"].float()

        kl = prior_log_scale - posterior_log_scale - 0.5
        kl = kl + 0.5 * (
            (z_prior - prior_mean).pow(2)
            * torch.exp(-2.0 * prior_log_scale)
        )

        return (kl * mask).sum() / mask.sum().clamp(min=1.0)

    def _tts010_mel_loss(self, outputs, spectrogram):
        a = self.audio
        segment_frames = outputs["generated_audio"].size(-1) // a["hop_length"]
        target_mel = linear_to_log_mel(spectrogram.float(), a["n_mels"], a["n_fft"], a["sample_rate"])
        target_mel = slice_segments_at(target_mel, outputs["start_frames"], segment_frames)
        generated_mel = waveform_to_log_mel(
            outputs["generated_audio"].float().squeeze(1), a["n_mels"], a["n_fft"],
            a["hop_length"], a["win_length"], a["sample_rate"])
        return F.l1_loss(generated_mel, target_mel)

    def forward(self, outputs, target_audio, spectrogram=None, gan_loss=None, fm_loss=None):
        generated_audio = outputs["generated_audio"].float().squeeze(1)
        target_audio = target_audio.float().squeeze(1)
        zero = generated_audio.new_tensor(0.0)

        if self.recipe == "tts010":
            if spectrogram is None:
                raise ValueError("The tts010 reconstruction loss needs the training spectrogram.")
            mel_loss = self._tts010_mel_loss(outputs, spectrogram)
            stft_loss = zero
            waveform_loss = zero
        else:
            mel_loss = F.l1_loss(
                torch.log(self.mel_transform(generated_audio).clamp(min=1e-5)),
                torch.log(self.mel_transform(target_audio).clamp(min=1e-5)),
            )
            spectral_convergence, log_magnitude_loss = self.stft_loss(generated_audio, target_audio)
            stft_loss = spectral_convergence + log_magnitude_loss
            waveform_loss = F.l1_loss(generated_audio, target_audio)

        duration_loss = self._duration_loss(outputs)
        kl_loss = self._kl_loss(outputs)
        gan_loss = zero if gan_loss is None else gan_loss.float()
        fm_loss = zero if fm_loss is None or not self.feature_matching else fm_loss.float()

        weighted = {
            "weighted_mel": self.mel_weight * mel_loss,
            "weighted_stft": self.stft_weight * stft_loss,
            "weighted_wave": self.wave_weight * waveform_loss,
            "weighted_duration": self.duration_weight * duration_loss,
            "weighted_kl": self.kl_weight * kl_loss,
            "weighted_gan": self.gan_weight * gan_loss,
            "weighted_feature_matching": self.feature_matching_weight * fm_loss,
        }
        return {
            "total": sum(weighted.values()),
            "mel": mel_loss,
            "stft": stft_loss,
            "wave": waveform_loss,
            "duration": duration_loss,
            "kl": kl_loss,
            "gan": gan_loss,
            "feature_matching": fm_loss,
            **weighted,
        }

In [ ]:
def get_padding(kernel_size, dilation=1):
    return int((kernel_size * dilation - dilation) / 2)


class DiscriminatorP(nn.Module):
    """Period discriminator: folds the waveform into [T / period, period] and applies 2-D convs."""

    def __init__(self, period, kernel_size=5, stride=3, use_spectral_norm=False):
        super().__init__()
        self.period = period
        self.use_spectral_norm = use_spectral_norm
        norm_f = spectral_norm if use_spectral_norm else weight_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(nn.Conv2d(1024, 1024, (kernel_size, 1), 1, padding=(get_padding(kernel_size, 1), 0))),
        ])
        self.conv_post = norm_f(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []

        # 1d to 2d
        b, c, t = x.shape
        if t % self.period != 0:  # pad first
            n_pad = self.period - (t % self.period)
            x = F.pad(x, (0, n_pad), "reflect")
            t = t + n_pad
        x = x.view(b, c, t // self.period, self.period)

        for layer in self.convs:
            x = layer(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap


class DiscriminatorS(nn.Module):
    """Scale discriminator on the raw waveform (grouped 1-D convs)."""

    def __init__(self, use_spectral_norm=False):
        super().__init__()
        norm_f = spectral_norm if use_spectral_norm else weight_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv1d(1, 16, 15, 1, padding=7)),
            norm_f(nn.Conv1d(16, 64, 41, 4, groups=4, padding=20)),
            norm_f(nn.Conv1d(64, 256, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(256, 1024, 41, 4, groups=64, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 41, 4, groups=256, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = norm_f(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []

        for layer in self.convs:
            x = layer(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap


class MultiPeriodDiscriminator(nn.Module):
    """TTS_0.10 discriminator (benlu): one scale + five period discriminators."""

    def __init__(self, use_spectral_norm=False):
        super().__init__()
        periods = [2, 3, 5, 7, 11]

        discs = [DiscriminatorS(use_spectral_norm=use_spectral_norm)]
        discs = discs + [DiscriminatorP(i, use_spectral_norm=use_spectral_norm) for i in periods]
        self.discriminators = nn.ModuleList(discs)

    def forward(self, y, y_hat):
        y_d_rs = []
        y_d_gs = []
        fmap_rs = []
        fmap_gs = []
        for d in self.discriminators:
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            y_d_gs.append(y_d_g)
            fmap_rs.append(fmap_r)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


class LegacyDiscriminator(nn.Module):
    """TTS_0.02 discriminator: one 1-D convolution stack, same parameters and outputs."""

    def __init__(
        self,
        base_channels=32,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=15, stride=1, padding=7),
            nn.LeakyReLU(0.2),
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=41, stride=4, padding=20),
            nn.LeakyReLU(0.2),
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=41, stride=4, padding=20),
            nn.LeakyReLU(0.2),
            nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=41, stride=4, padding=20),
            nn.LeakyReLU(0.2),
            nn.Conv1d(base_channels * 8, base_channels * 16, kernel_size=41, stride=4, padding=20),
            nn.LeakyReLU(0.2),
            nn.Conv1d(base_channels * 16, 1, kernel_size=5, stride=1, padding=2),
        )

    def score(self, waveform):
        """Scores and feature maps (each LeakyReLU output, then the final scores)."""
        fmap = []
        x = waveform
        for layer in self.network:
            x = layer(x)
            if isinstance(layer, nn.LeakyReLU):
                fmap.append(x)
        fmap.append(x)
        return x, fmap

    def forward(self, y, y_hat):
        real, fmap_r = self.score(y)
        fake, fmap_g = self.score(y_hat)
        return [real], [fake], [fmap_r], [fmap_g]

## 14. Trainer

Each step updates the discriminator on real versus detached generated audio, then the
generator with the discriminator frozen (the same order as benlu's trainer). With
`optim = tts010` the generator and discriminator forward passes run under fp16 autocast and
one `GradScaler` scales both updates; with `optim = tts002` everything runs in fp32 and
gradients are clipped at norm 1.0. Gradient norms are logged in both cases.

A step whose loss is not finite is skipped and counted (`nonfinite_skips`); 20 consecutive
skips stop the run. Under autocast the scaler also skips steps with overflowing gradients
(`amp_backoffs`). Checkpoints contain both optimizers, the scaler, the switches and the
speaker mapping; resuming with different switches or a different mapping is refused.

In [ ]:
class VITSTrainer:
    MAX_CONSECUTIVE_NONFINITE = 20

    def __init__(
        self,
        model,
        discriminator,
        loss_function,
        train_loader,
        learning_rate=2e-4,
        discriminator_learning_rate=None,
        use_amp=False,
        max_grad_norm=None,
        checkpoint_dir="./model_weight",
        sample_dir="./generated_samples",
        device=None,
        speaker_to_id=None,
        switches=None,
        model_version="TTS_0.10",
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        if discriminator_learning_rate is None:
            discriminator_learning_rate = learning_rate

        self.speaker_to_id = dict(speaker_to_id or {})
        if (len(self.speaker_to_id) != model.n_speakers
                or any(not isinstance(name, str) or not name.strip() for name in self.speaker_to_id)
                or any(type(value) is not int for value in self.speaker_to_id.values())
                or sorted(self.speaker_to_id.values()) != list(range(model.n_speakers))):
            raise ValueError("speaker_to_id must name each configured speaker exactly once.")
        self.device = torch.device(device)
        self.use_amp = bool(use_amp) and self.device.type == "cuda"
        self.max_grad_norm = max_grad_norm
        self.switches = dict(switches or {})
        self.model_version = model_version
        self.train_loader = train_loader

        self.model = model.to(self.device)
        self.base_model = self.model  # evaluation interface
        self.discriminator = discriminator.to(self.device)
        self.loss_function = loss_function.to(self.device)
        print(f"Training on one device: {self.device}; fp16 autocast: {self.use_amp}")

        self.generator_optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=learning_rate,
            betas=(0.8, 0.99),
            eps=1e-9,
            weight_decay=0.01,
        )
        self.optimizer = self.generator_optimizer
        self.discriminator_optimizer = torch.optim.AdamW(
            self.discriminator.parameters(),
            lr=discriminator_learning_rate,
            betas=(0.8, 0.99),
            eps=1e-9,
            weight_decay=0.01,
        )
        self.scaler = torch.amp.GradScaler(self.device.type, enabled=self.use_amp)

        # Counters retained in checkpoints and status records.
        self.optimizer_update_stats = {"updates": 0}
        self.counters = {"nonfinite_skips": 0, "amp_backoffs": 0}
        self._consecutive_nonfinite = 0
        self.step = 0
        self.init_from = None  # source checkpoint of a run started with init_from_checkpoint

        self.checkpoint_dir = Path(checkpoint_dir)
        self.sample_dir = Path(sample_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.sample_dir.mkdir(parents=True, exist_ok=True)

    def _move_batch_to_device(self, batch):
        return tuple(
            item.to(self.device, non_blocking=True)
            for item in batch
        )

    def speaker_config(self):
        return {"n_speakers": self.base_model.n_speakers,
                "gin_channels": self.base_model.gin_channels,
                "speaker_to_id": dict(self.speaker_to_id)}

    def save_checkpoint(self):
        checkpoint = {
            "model_version": self.model_version,
            "switches": dict(self.switches),
            "speaker_config": self.speaker_config(),
            "step": self.step,
            "optimizer_updates": self.optimizer_update_stats["updates"],
            "counters": dict(self.counters),
            "model": self.base_model.state_dict(),
            "optimizer": self.generator_optimizer.state_dict(),
            "discriminator": self.discriminator.state_dict(),
            "discriminator_optimizer": self.discriminator_optimizer.state_dict(),
            "scaler": self.scaler.state_dict(),
        }
        if self.init_from:
            checkpoint["init_from"] = dict(self.init_from)

        path = self.checkpoint_dir / f"vits_step_{self.step}.pt"
        torch.save(checkpoint, path)
        print(f"Saved checkpoint: {path}")

    def _load_compatible(self, path, action):
        checkpoint = torch.load(path, map_location=self.device, weights_only=True)
        stored = checkpoint.get("speaker_config", {
            "n_speakers": 0, "gin_channels": 0, "speaker_to_id": {}})
        if stored != self.speaker_config():
            raise ValueError(f"Checkpoint speaker configuration/mapping differs; {action} refused.")
        if checkpoint.get("switches") != self.switches:
            raise ValueError(f"Checkpoint switches {checkpoint.get('switches')} differ from "
                             f"{self.switches}; {action} refused.")
        return checkpoint

    def init_from_checkpoint(self, path, carry_optimizer=True):
        """Start a new run from a finished checkpoint of the same arm (not a resume).

        Loads generator and discriminator weights and, with carry_optimizer, both optimizers'
        moments and the fp16 scaler; the step and update counters stay at 0.
        """
        import hashlib
        checkpoint = self._load_compatible(path, "initialisation")
        self.base_model.load_state_dict(checkpoint["model"])
        self.discriminator.load_state_dict(checkpoint["discriminator"])
        if carry_optimizer:
            if "optimizer" not in checkpoint:
                raise ValueError(f"{path} has no optimizer state; start with fresh optimizers instead.")
            self.generator_optimizer.load_state_dict(checkpoint["optimizer"])
            self.discriminator_optimizer.load_state_dict(checkpoint["discriminator_optimizer"])
            self.scaler.load_state_dict(checkpoint["scaler"])
        self.init_from = {
            "checkpoint": str(Path(path).resolve()),
            "sha256": hashlib.sha256(Path(path).read_bytes()).hexdigest(),
            "step": checkpoint.get("step"),
            "model_version": checkpoint.get("model_version"),
            "optimizer": "carried" if carry_optimizer else "fresh",
            "source_init_from": checkpoint.get("init_from"),
        }
        print(f"Initialised from {Path(path).name} (step {checkpoint.get('step')}), "
              f"optimizer state {self.init_from['optimizer']}; this run starts at step 0.")
        return dict(self.init_from)

    def load_checkpoint(self, path):
        checkpoint = self._load_compatible(path, "resume")
        self.base_model.load_state_dict(checkpoint["model"])
        self.generator_optimizer.load_state_dict(checkpoint["optimizer"])
        self.discriminator.load_state_dict(checkpoint["discriminator"])
        self.discriminator_optimizer.load_state_dict(checkpoint["discriminator_optimizer"])
        self.scaler.load_state_dict(checkpoint["scaler"])
        self.step = checkpoint["step"]
        self.optimizer_update_stats["updates"] = checkpoint.get("optimizer_updates")
        self.counters.update(checkpoint.get("counters", {}))
        self.init_from = checkpoint.get("init_from")
        print(f"Resumed from step {self.step}")

    @torch.inference_mode()
    def save_samples(self, batch, number_of_samples=2):
        was_training = self.model.training
        self.model.eval()

        text_tokens, text_lengths = batch[0], batch[1]
        text_tokens = text_tokens[:number_of_samples]
        text_lengths = text_lengths[:number_of_samples]

        result = self.base_model.infer(
            text_tokens,
            text_lengths,
            noise_scale=0.667,
            max_frames=1000,
            speaker_ids=batch[6][:number_of_samples] if len(batch) == 7 else None,
        )

        for index in range(text_tokens.size(0)):
            audio_length = int(result["audio_lengths"][index].item())
            waveform = result["generated_audio"][
                index,
                :,
                :audio_length,
            ].float().cpu().clamp(-1.0, 1.0)

            path = self.sample_dir / (
                f"step_{self.step}_sample_{index + 1}.wav"
            )

            torchaudio.save(str(path), waveform, sample_rate=22050)

        self.model.train(was_training)

    def _set_discriminator_requires_grad(self, requires_grad):
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(requires_grad)

    def _target_audio_from_outputs(self, outputs, waveform):
        target_start_samples = (
            outputs["start_frames"] * self.base_model.hop_length
        )

        return slice_segments_at(
            waveform,
            target_start_samples,
            self.base_model.segment_samples,
        )

    def _autocast(self):
        return torch.autocast(self.device.type, dtype=torch.float16, enabled=self.use_amp)

    def _update(self, loss, optimizer, parameters):
        """Scaled backward, optional clipping and step. Returns the pre-clipping gradient norm."""
        optimizer.zero_grad(set_to_none=True)
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(optimizer)
        limit = self.max_grad_norm if self.max_grad_norm is not None else float("inf")
        norm = torch.nn.utils.clip_grad_norm_(parameters, limit)
        self.scaler.step(optimizer)
        return norm.detach()

    def _note_nonfinite(self, what):
        self.counters["nonfinite_skips"] += 1
        self._consecutive_nonfinite += 1
        print(f"Non-finite {what} loss at step {self.step + 1}; update skipped.")
        if self._consecutive_nonfinite >= self.MAX_CONSECUTIVE_NONFINITE:
            raise FloatingPointError(
                f"{self._consecutive_nonfinite} consecutive non-finite steps at step {self.step + 1}.")

    def _training_step(self, batch):
        if len(batch) != (7 if self.base_model.n_speakers else 6):
            raise ValueError("Batch conditioning does not match model configuration.")
        (
            text_tokens,
            text_lengths,
            spectrogram,
            spectrogram_lengths,
            waveform,
            _,
        ) = batch[:6]

        with self._autocast():
            outputs = self.model(
                text_tokens,
                text_lengths,
                spectrogram,
                spectrogram_lengths,
                speaker_ids=batch[6] if len(batch) == 7 else None,
            )
        generated_audio = outputs["generated_audio"]
        target_audio = self._target_audio_from_outputs(outputs, waveform)
        stepped = False
        nan = generated_audio.new_tensor(float("nan")).float()

        # Discriminator update on real vs detached generated audio.
        with self._autocast():
            real_scores, fake_scores, _, _ = self.discriminator(target_audio, generated_audio.detach())
        discriminator_losses = discriminator_lsgan_loss(real_scores, fake_scores)
        discriminator_norm = nan
        if torch.isfinite(discriminator_losses["total"]):
            discriminator_norm = self._update(discriminator_losses["total"], self.discriminator_optimizer,
                                              self.discriminator.parameters())
            stepped = True
        else:
            self._note_nonfinite("discriminator")

        # Generator update; discriminator parameters are frozen but gradients flow through it.
        generator_norm = nan
        self._set_discriminator_requires_grad(False)
        try:
            with self._autocast():
                _, fake_scores_g, real_maps, fake_maps = self.discriminator(target_audio, generated_audio)
            generator_gan = generator_lsgan_loss(fake_scores_g)
            fm = feature_matching_loss(real_maps, fake_maps) if self.loss_function.feature_matching else None
            losses = self.loss_function(outputs, target_audio, spectrogram=spectrogram,
                                        gan_loss=generator_gan, fm_loss=fm)
            if torch.isfinite(losses["total"]):
                generator_norm = self._update(losses["total"], self.generator_optimizer, self.model.parameters())
                stepped = True
                if torch.isfinite(discriminator_losses["total"]):
                    self._consecutive_nonfinite = 0
            else:
                self._note_nonfinite("generator")
        finally:
            self._set_discriminator_requires_grad(True)

        scale = self.scaler.get_scale() if self.use_amp else 1.0
        if stepped:
            self.scaler.update()
            if self.use_amp and self.scaler.get_scale() < scale:
                self.counters["amp_backoffs"] += 1
            elif self.optimizer_update_stats["updates"] is not None and torch.isfinite(losses["total"]):
                self.optimizer_update_stats["updates"] += 1

        losses = dict(losses)
        losses.update({
            "gan_discriminator": discriminator_losses["total"].detach(),
            "gan_discriminator_real": discriminator_losses["real"].detach(),
            "gan_discriminator_fake": discriminator_losses["fake"].detach(),
            "discriminator_real_score": torch.stack([s.detach().float().mean() for s in real_scores]).mean(),
            "discriminator_fake_score": torch.stack([s.detach().float().mean() for s in fake_scores]).mean(),
            "grad_norm_generator": generator_norm,
            "grad_norm_discriminator": discriminator_norm,
            "amp_scale": generated_audio.new_tensor(float(scale)).float(),
        })
        return outputs, target_audio, losses

In [ ]:
# Full-dataset training loop with the records needed for the ablation report.
import json
import shutil
import time
from datetime import datetime, timezone


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


def finite_or_none(value):
    """JSON has no NaN/Infinity: record non-finite diagnostics (e.g. overflowing gradients) as null."""
    value = float(value)
    return value if math.isfinite(value) else None


def checkpoint_step(path):
    return int(Path(path).stem.rsplit("_", 1)[1])


def prune_checkpoints(checkpoint_dir, keep_every, latest_step):
    """Keep milestone checkpoints (multiples of keep_every) and the latest one."""
    for path in Path(checkpoint_dir).glob("vits_step_*.pt"):
        step = checkpoint_step(path)
        if step != latest_step and step % keep_every != 0:
            path.unlink()


def run_full_training(trainer, evaluators, run_dir, *, total_steps, eval_every, save_every,
                      keep_every, log_every, sample_batch, time_budget_hours, history=None):
    """Return (training_history, stop_reason). Stops cleanly before the time budget.

    evaluators: {name: ReconstructionEvaluator}; each runs every eval_every steps.
    """
    run_dir = Path(run_dir)
    history = list(history or [])
    trainer.model.train()
    trainer.discriminator.train()
    sample_batch = trainer._move_batch_to_device(sample_batch)
    started = time.perf_counter()
    deadline = started + time_budget_hours * 3600
    window_start, window_step = started, trainer.step

    def save(step):
        trainer.save_checkpoint()
        prune_checkpoints(trainer.checkpoint_dir, keep_every, step)
        try:
            trainer.save_samples(sample_batch)  # text-only inference for the first two probe texts
        except Exception as error:  # samples are for listening only; never stop training
            print(f"Sample generation failed at step {step}: {error!r}")

    def evaluate(step, save_audio):
        for name, evaluator in evaluators.items():
            summary = evaluator.evaluate(trainer.base_model, step, save_audio=save_audio)
            print(f"{name} evaluation at {step}:",
                  {row["mode"]: round(row["log_mel_l1"], 4) for row in summary}, flush=True)

    if trainer.step == 0:
        # Save audio at step 0 so an audio-backend problem fails in minutes, not hours.
        evaluate(0, save_audio=True)

    stop_reason = "completed"
    while trainer.step < total_steps and stop_reason == "completed":
        for batch in trainer.train_loader:
            if trainer.step >= total_steps:
                break
            batch = trainer._move_batch_to_device(batch)
            _, _, losses = trainer._training_step(batch)
            trainer.step += 1
            step = trainer.step

            if step % log_every == 0 or step == total_steps:
                now = time.perf_counter()
                rate = (now - window_start) / max(step - window_step, 1)
                window_start, window_step = now, step
                row = {
                    "step": step,
                    "optimizer_updates": trainer.optimizer_update_stats["updates"],
                    **trainer.counters,
                    "sec_per_iter": rate,
                    "session_elapsed_sec": now - started,
                    **{name: finite_or_none(value.detach().item()) for name, value in losses.items()},
                }
                history.append(row)
                write_json(run_dir / "training_history.json", history)
                if step % (log_every * 10) == 0 or step == total_steps:
                    eta = rate * (total_steps - step) / 3600
                    shown = {key: row[key] for key in ("total", "mel", "kl", "duration", "gan",
                                                       "feature_matching", "gan_discriminator")}
                    print(f"step {step}/{total_steps}  "
                          + "  ".join(f"{k} {v:.3f}" if v is not None else f"{k} n/a" for k, v in shown.items())
                          + f"  {rate:.3f} s/it  ETA {eta:.2f} h", flush=True)

            if step % eval_every == 0 or step == total_steps:
                evaluate(step, save_audio=step % keep_every == 0 or step == total_steps)

            if step % save_every == 0 or step == total_steps:
                save(step)

            if time.perf_counter() > deadline and step < total_steps:
                if step % save_every != 0:
                    save(step)
                stop_reason = "stopped_time_budget"
                print(f"Time budget reached at step {step}; resume from the latest checkpoint.")
                break
    return history, stop_reason

In [ ]:
# Shared evaluation implementation lives outside this notebook.
import sys
import hashlib

sys.path.insert(0, str(REPO_ROOT))
from sts_eval import EvaluationConfig, ReconstructionEvaluator, plot_history
from sts_eval.reconstruction import repository_provenance

# ---- Run settings. Identical for every arm; only the arm's switches and seed differ. ----
TOTAL_STEPS = int(os.environ.get("STS_TOTAL_STEPS", "40000"))
EVAL_EVERY = int(os.environ.get("STS_EVAL_EVERY", "2000"))
SAVE_EVERY = int(os.environ.get("STS_SAVE_EVERY", "5000"))     # latest checkpoint for resume
KEEP_EVERY = int(os.environ.get("STS_KEEP_EVERY", "10000"))    # milestone checkpoints kept
LOG_EVERY = int(os.environ.get("STS_LOG_EVERY", "100"))
# Kaggle stops sessions at 12 h. Stop cleanly before that so the output is saved.
TIME_BUDGET_HOURS = float(os.environ.get("STS_TIME_BUDGET_HOURS", "11.3"))
# To continue a stopped run: set this to its run folder.
RESUME_FROM = os.environ.get("STS_RESUME_FROM") or None
# To start a NEW run (e.g. a fine-tune) from a finished checkpoint of this arm: set its path.
INIT_FROM = os.environ.get("STS_INIT_FROM") or None
INIT_OPTIMIZER = os.environ.get("STS_INIT_OPTIMIZER", "carry")  # "carry" or "fresh"
if RESUME_FROM and INIT_FROM:
    raise ValueError("Set STS_RESUME_FROM or STS_INIT_FROM, not both: a resumed run keeps its initialisation.")
if INIT_OPTIMIZER not in ("carry", "fresh"):
    raise ValueError(f"STS_INIT_OPTIMIZER must be carry or fresh, not {INIT_OPTIMIZER!r}.")
RUN_TAG = os.environ.get("STS_RUN_TAG", "")
if RUN_TAG and not all(c.isalnum() or c == "-" for c in RUN_TAG):
    raise ValueError("STS_RUN_TAG may contain only letters, digits and '-'.")
EVAL_SEED = 1234
MODEL_VARIANT = f"tts010_{ARM}" + (f"_{RUN_TAG}" if RUN_TAG else "")

# Probe 1: the eight LJ001 utterances of the earlier protocols. Under the length filter,
# LJ001-0001/0003/0005/0007 (more than 100 characters) are not trained on.
FIXED_IDS = [f"LJ001-{i:04d}" for i in range(1, 9)]
if NUM_SPEAKERS:
    # One real training utterance per speaker, up to eight; not held-out.
    FIXED_IDS = metadata_df.groupby("speaker_id", sort=True).head(1)["file_name"].tolist()[:8]
indexed_corpus = corpus_df.set_index("file_name", drop=False)
fixed_metadata = indexed_corpus.loc[FIXED_IDS].reset_index(drop=True)
fixed_dataset = LJSpeechDataset(fixed_metadata, WAV_DIR, SAMPLE_RATE, N_FFT, HOP_LENGTH, WIN_LENGTH)
fixed_batch = collate_tts_batch([fixed_dataset[i] for i in range(len(fixed_dataset))])
TRAINING_IDS = set(metadata_df["file_name"])
probe_membership = {name: ("trained" if name in TRAINING_IDS else "not_trained") for name in FIXED_IDS}

# Probe 2: the first eight held-out clips (never trained on in any arm).
HELDOUT_PROBE_IDS = [name for name in heldout_df["file_name"] if name in set(HELDOUT_BASE_IDS)][:8]

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path(os.environ.get("STS_RUN_ROOT", str(REPO_ROOT / "output" / "full-train"))) / f"{MODEL_VARIANT}_{run_id}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

evaluation_config = EvaluationConfig(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH, segment_frames=SEGMENT_FRAMES, seed=EVAL_SEED,
)
notebook_file = REPO_ROOT / "notebooks" / "vits-full-train.ipynb"
run_config = {
    "model_version": MODEL_VERSION,
    "arm": ARM,
    "arm_title": ARMS[ARM]["title"],
    "reverted_switches": ARMS[ARM]["revert"],
    "switches": SWITCHES,
    "optim_preset": OPTIM,
    "autocast_fp16": USE_AMP,
    "speaker_config": {"n_speakers": NUM_SPEAKERS, "gin_channels": GIN_CHANNELS,
                       "speaker_to_id": SPEAKER_TO_ID},
    "variant": MODEL_VARIANT, "train_seed": TRAIN_SEED,
    "total_steps": TOTAL_STEPS, "eval_every": EVAL_EVERY, "save_every": SAVE_EVERY,
    "keep_every": KEEP_EVERY, "log_every": LOG_EVERY, "time_budget_hours": TIME_BUDGET_HOURS,
    "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "chunk_training": CHUNK_TRAINING, "init_from": None,
    "corpus_clips": len(corpus_df), "training_clips": len(metadata_df),
    "heldout_ids": HELDOUT_IDS,
    "heldout_split_sha256": hashlib.sha256(HELDOUT_PATH.read_bytes()).hexdigest() if HELDOUT_IDS else None,
    "heldout_extra": {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in HELDOUT_EXTRA_PATHS},
    "phoneme_cache": ({"path": str(PHONEME_CACHE), "sha256": hashlib.sha256(PHONEME_CACHE.read_bytes()).hexdigest()}
                      if PHONEME_CACHE else None),
    "probe_sample_ids": FIXED_IDS, "probe_membership": probe_membership,
    "heldout_probe_ids": HELDOUT_PROBE_IDS,
    "data_dir": str(DATA_DIR),
    "metadata_sha256": hashlib.sha256(METADATA_PATH.read_bytes()).hexdigest(),
    "notebook_sha256": hashlib.sha256(notebook_file.read_bytes()).hexdigest() if notebook_file.is_file() else None,
    "eval_source_sha256": hashlib.sha256((REPO_ROOT / "sts_eval" / "reconstruction.py").read_bytes()).hexdigest(),
    "repository": repository_provenance(REPO_ROOT),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "torch": torch.__version__, "torch_cuda": torch.version.cuda,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "mas": "batch-vectorised NumPy; identical paths to the per-item loop",
    "scope": ("full manifest training; fixed-crop probe on training utterances; not held-out" if NUM_SPEAKERS
              else f"LJSpeech minus {len(HELDOUT_IDS)} held-out clips; fixed-crop probes only; no text-to-speech metric"),
}
evaluators = {"probe": ReconstructionEvaluator(
    fixed_batch, FIXED_IDS, RUN_DIR / "evaluation",
    config=evaluation_config, provenance=run_config,
    scope="fixed_lj001_probe_" + ("training_utterances" if NUM_SPEAKERS else "mixed_trained_and_length_filtered"),
)}
if HELDOUT_PROBE_IDS:
    heldout_probe_frame = heldout_df.set_index("file_name", drop=False).loc[HELDOUT_PROBE_IDS].reset_index(drop=True)
    heldout_probe_dataset = LJSpeechDataset(heldout_probe_frame, WAV_DIR, SAMPLE_RATE, N_FFT, HOP_LENGTH, WIN_LENGTH)
    heldout_probe_batch = collate_tts_batch([heldout_probe_dataset[i] for i in range(len(heldout_probe_dataset))])
    evaluators["heldout"] = ReconstructionEvaluator(
        heldout_probe_batch, HELDOUT_PROBE_IDS, RUN_DIR / "evaluation_heldout",
        config=evaluation_config, provenance=run_config, scope="held_out_not_trained",
    )
else:
    print("No held-out recordings available: held-out probe disabled.")
print("Probe membership:", probe_membership)
print("Run artifacts:", RUN_DIR)

## 15. Build the model and trainer

The arm's switches choose each component, the discriminator, the loss recipe and the
optimisation preset. A resumed run must have the same arm settings, data and split.

`STS_INIT_FROM` starts a **new** run from a finished checkpoint of the same arm (for example a
fine-tune of R0 at 40,000 steps); this is not a resume. It loads the generator and discriminator
weights and, by default, both optimizers' moments and the fp16 scaler (`STS_INIT_OPTIMIZER=fresh`
starts them anew). The step counter starts at 0, so `STS_TOTAL_STEPS` counts fine-tuning steps.
The source checkpoint (path, SHA-256, step) is stored in `run_config.json` and in every
checkpoint of the new run. A stopped fine-tune is continued with `STS_RESUME_FROM` in place of
`STS_INIT_FROM` and the same steps and chunk settings; it keeps the original `init_from`.

In [ ]:
model = VITSModel(
    vocab_size=len(symbol_to_id),
    padding_id=PAD_ID,
    spec_channels=N_FFT // 2 + 1,
    latent_channels=192,
    hidden_channels=192,
    feedforward_channels=768,
    decoder_channels=512,
    segment_samples=SEGMENT_SAMPLES,
    hop_length=HOP_LENGTH,
    n_speakers=NUM_SPEAKERS,
    gin_channels=GIN_CHANNELS,
    text_encoder=SWITCHES["text_encoder"],
    posterior_encoder=SWITCHES["posterior_encoder"],
    decoder=SWITCHES["decoder"],
    duration_predictor=SWITCHES.get("duration_predictor", "deterministic"),
    encoder_window=SWITCHES.get("encoder_window"),
    space_id=symbol_to_id[" "],
)

if SWITCHES["discriminator"] == "tts010":
    discriminator = MultiPeriodDiscriminator()
else:
    discriminator = LegacyDiscriminator(base_channels=32)

loss_function = VITSLoss(
    recipe=SWITCHES["recon_loss"],
    feature_matching=SWITCHES["feature_matching"],
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
)

trainer = VITSTrainer(
    model=model,
    discriminator=discriminator,
    loss_function=loss_function,
    train_loader=train_loader,
    learning_rate=OPTIM["learning_rate"],
    use_amp=USE_AMP,
    max_grad_norm=OPTIM["max_grad_norm"],
    checkpoint_dir=RUN_DIR / "checkpoints",
    sample_dir=RUN_DIR / "samples",
    speaker_to_id=SPEAKER_TO_ID,
    switches=SWITCHES,
    model_version=MODEL_VERSION,
)

run_config["model"] = repr(model)
run_config["discriminator"] = repr(discriminator)
run_config["parameters"] = {
    "generator": sum(p.numel() for p in model.parameters()),
    "discriminator": sum(p.numel() for p in discriminator.parameters()),
}
run_config["optimizer"] = {k: v for k, v in trainer.generator_optimizer.param_groups[0].items() if k != "params"}
run_config["loss_weights"] = loss_function.weights()

# A new run started from a finished checkpoint: its weights (and optimizer state), step 0.
if INIT_FROM:
    run_config["init_from"] = trainer.init_from_checkpoint(INIT_FROM, carry_optimizer=INIT_OPTIMIZER == "carry")


def chunk_setting(config):
    """Chunk-training settings without the machine-specific path of the durations file."""
    chunk = config.get("chunk_training")
    return chunk and {**chunk, "durations": {k: v for k, v in chunk["durations"].items() if k != "path"}}


# Resume a run that stopped at the time budget: copy its records, load its latest checkpoint.
training_history = []
if RESUME_FROM:
    previous = Path(RESUME_FROM)
    previous_config = json.loads((previous / "run_config.json").read_text(encoding="utf-8"))
    for key in ("metadata_sha256", "heldout_split_sha256", "probe_sample_ids", "arm", "switches",
                "total_steps", "train_seed", "batch_size", "loss_weights"):
        if previous_config.get(key) != run_config[key]:
            raise ValueError(f"Cannot resume: {key} differs from the previous run.")
    if chunk_setting(previous_config) != chunk_setting(run_config):
        raise ValueError("Cannot resume: chunk_training differs from the previous run.")
    run_config["init_from"] = previous_config.get("init_from")
    latest = max((previous / "checkpoints").glob("vits_step_*.pt"), key=checkpoint_step)
    trainer.load_checkpoint(latest)
    shutil.copytree(previous / "checkpoints", trainer.checkpoint_dir, dirs_exist_ok=True)
    if (previous / "samples").is_dir():
        shutil.copytree(previous / "samples", trainer.sample_dir, dirs_exist_ok=True)
    training_history = [row for row in json.loads((previous / "training_history.json").read_text(encoding="utf-8"))
                        if row["step"] <= trainer.step]
    for evaluator in evaluators.values():
        previous_dir = previous / evaluator.output_dir.name
        previous_eval = json.loads((previous_dir / "evaluation_history.json").read_text(encoding="utf-8"))
        evaluator.history = [row for row in previous_eval if row["step"] <= trainer.step]
        evaluator._last_step = trainer.step
        for step_dir in previous_dir.glob("step_*"):
            if int(step_dir.name.split("_")[1]) <= trainer.step:
                shutil.copytree(step_dir, evaluator.output_dir / step_dir.name, dirs_exist_ok=True)
    run_config["resumed_from"] = {"run_dir": str(previous), "checkpoint": latest.name, "step": trainer.step,
                                  "previous_resumes": previous_config.get("resumed_from")}
    print(f"Resumed {previous.name} at step {trainer.step}. Data order after a resume differs "
          "from an uninterrupted run.")

write_json(RUN_DIR / "run_config.json", run_config)
print(run_config["parameters"], run_config["loss_weights"])

In [ ]:
# Runs training. Do not execute this cell just to check installation.
session_start = time.perf_counter()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)
try:
    training_history, stop_reason = run_full_training(
        trainer, evaluators, RUN_DIR,
        total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY, save_every=SAVE_EVERY,
        keep_every=KEEP_EVERY, log_every=LOG_EVERY, sample_batch=fixed_batch,
        time_budget_hours=TIME_BUDGET_HOURS, history=training_history,
    )
except BaseException as error:
    write_json(RUN_DIR / "status.json", {
        "status": "failed_or_interrupted", "arm": ARM, "step": trainer.step,
        "ended_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "error_type": type(error).__name__, "error": str(error),
        **trainer.counters,
    })
    raise
else:
    ended = datetime.now(timezone.utc).isoformat(timespec="seconds")
    status = {
        "status": stop_reason, "arm": ARM, "step": trainer.step, "total_steps": TOTAL_STEPS,
        "actual_optimizer_updates": trainer.optimizer_update_stats["updates"],
        **trainer.counters,
        "session_wall_seconds": round(time.perf_counter() - session_start, 1),
        "resumed": bool(RESUME_FROM),
        "completion_time_source": "runtime_clock",
        "gpu": run_config["gpu"],
        "peak_allocated_GiB": round(torch.cuda.max_memory_allocated() / 2**30, 3) if DEVICE.type == "cuda" else None,
    }
    status["completed_at_utc" if stop_reason == "completed" else "ended_at_utc"] = ended
    write_json(RUN_DIR / "status.json", status)
print("Experiment record:", (RUN_DIR / "status.json").read_text(encoding="utf-8"))

In [ ]:
# Curves, the last probe metrics and the scored audio of the final evaluation.
import matplotlib.pyplot as plt

figure = plot_history(evaluators["probe"].history, training_history, RUN_DIR / "curves.png")
plt.show()
for name, evaluator in evaluators.items():
    print(name)
    print(pd.DataFrame(evaluator.history).tail(2).to_string(index=False))
try:
    from IPython import get_ipython
    from IPython.display import Audio, display
    if get_ipython() is None:  # headless run (tools/train_tts010.py)
        display = None
except ImportError:
    display = None
audio_dir = evaluators["probe"].output_dir / f"step_{trainer.step:06d}"
for name in ("target", "mean", "sample"):
    path = audio_dir / f"000_{name}.wav"
    if display is not None and path.is_file():
        print(FIXED_IDS[0], name)
        display(Audio(filename=str(path)))
print("Upload this folder to the team Google Drive:", RUN_DIR)